# FC Hradec Králové vs FC Viktoria Plzeň — Pre-Match Preview

Chance Liga 2026/27, matchday 5 (2026-08-23, 17:00 local, FINEP Arena, Hradec Králové). Built ahead of kickoff -- the fixture has not been played yet, so every page is built from each team's SEASON-TO-DATE totals (every match played so far, pooled together), not a single matchday snapshot or a head-to-head.

FC Hradec Králové: 3 matches played, 2W 1D 0L (won 2-1 at home over Pardubice, drew 0-0 away at Bohemians 1905, won 2-1 at home over Baník Ostrava). FC Viktoria Plzeň: 2 matches played, 0W 1D 1L (lost 1-3 at home to Slovan Liberec, drew 1-1 at home with Zbrojovka Brno) -- Plzeň's own matchday-3 fixture has not been played yet, so their total stays at 2 matches.

Shots are scored with the repo's real trained xG model (`Model/model_xg.pkl`, via `Model/xg_model.py`). This notebook inlines `match_data.py`, `build_charts.py` and `build_pdf.py` end to end and reproduces all 50 `Visuals/*.png` pages plus the compiled PDF.

## Setup

In [1]:
import json
import math
import os
import sys

# Notebook-safe path resolution (no __file__ inside a notebook cell) --
# assumes this notebook is opened/run from its own directory, same as
# REPO_ROOT in the original scripts.
NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(NOTEBOOK_DIR)))
EVENTS_DIR = os.path.join(REPO_ROOT, "CZ Events", "CZ 2026-2027")
sys.path.insert(0, REPO_ROOT)
from Model.xg_model import XGModel

_XG_MODEL = XGModel.load()

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Data loading + parsing (`match_data.py`)

Opta MA3 event feed, same typeId/qualifierId conventions as the rest of this repo. Team IDs cross-checked against goal-scorer counts (Hradec-Pardubice 2-1, Bohemians-Hradec 0-0, Hradec-Ostrava 2-1) and against `CZ 2026-2027 Matches.csv`.

`compute_attack_directions()` is disabled (returns `{}`) -- the per-half flip heuristic was found to corrupt shot locations for two of the teams pooled into this report (Plzeň's matchday-1/2 games, in the sibling Teplice vs Viktoria Plzen report), the same failure mode discovered in the Hradec Kralove vs Besiktas report. Re-verified for this report's own fixture teams (Hradec's 3 matches, Plzeň's 2). Shots are scored with the real trained xG model.

In [2]:
BOHEMIANS_ID = "bqcrqg0367eqzrt4vjb5apu6g"
HRADEC_ID = "1v75g4bk8vzrvu0jmaro6lila"
TEPLICE_ID = "41eivtin75c5fu33x3zfx956b"
PARDUBICE_ID = "4xbgquadoen1b303u4hi9nhg9"
OSTRAVA_ID = "dfvvrv84skv23rsn1k6kt4slc"
ZLIN_ID = "aj1nbeiatqrs6e47mnhjidn15"
ARTIS_BRNO_ID = "6onh4wiqdb8r50qa6oyxlbafg"
MLADA_BOLESLAV_ID = "2qui2adwsi022vu84b8gdw4z0"

PLZEN_ID = "c6fx1460nlkawjgh67sp7a1hd"
LIBEREC_ID = "2c4rs2vp0tyjiqa7y3gfttf24"
ZBROJOVKA_ID = "6k350zwynsc23f0sxw9akgc6y"

TEAM_NAMES = {
    HRADEC_ID: "FC Hradec Králové",
    PLZEN_ID: "FC Viktoria Plzeň",
    PARDUBICE_ID: "FK Pardubice",
    BOHEMIANS_ID: "Bohemians 1905",
    OSTRAVA_ID: "FC Baník Ostrava",
    LIBEREC_ID: "FC Slovan Liberec",
    ZBROJOVKA_ID: "FC Zbrojovka Brno",
}
TEAM_SHORT = {
    HRADEC_ID: "Hradec Kr.",
    PLZEN_ID: "Plzeň",
    PARDUBICE_ID: "Pardubice",
    BOHEMIANS_ID: "Bohemians",
    OSTRAVA_ID: "Ostrava",
    LIBEREC_ID: "Liberec",
    ZBROJOVKA_ID: "Zbrojovka Brno",
}

# This fixture (not yet played)
FIXTURE_HOME_ID, FIXTURE_AWAY_ID = HRADEC_ID, PLZEN_ID
FIXTURE_HOME_NAME, FIXTURE_AWAY_NAME = TEAM_NAMES[HRADEC_ID], TEAM_NAMES[PLZEN_ID]
COMPETITION = "Chance Liga 2026/27, Matchday 5"
VENUE = "FINEP Arena, Hradec Králové"
MATCH_DATE = "2026-08-23"
KICKOFF_LOCAL = "17:00"
SOURCE = "Opta event data (every match each team has played this season) + trained xG model (Model/model_xg.pkl)"

X_SCALE, Y_SCALE = 1.05, 0.68     # Opta 0-100 units -> metres (105 x 68 pitch)
GOAL_X = 105.0
GOAL_Y = 34.0
GOAL_WIDTH = 7.32

T_PASS, T_TAKE_ON, T_FOUL, T_OUT = 1, 3, 4, 5
T_CORNER_AWARDED = 6
T_TACKLE, T_INTERCEPTION = 7, 8
T_CLEARANCE = 12
T_MISS, T_ATTEMPT_SAVED, T_GOAL, T_POST = 13, 15, 16, 14
T_CARD = 17
T_SUB_OFF, T_SUB_ON = 18, 19
T_CHALLENGE = 45
T_AERIAL = 44
T_BALL_RECOVERY, T_DISPOSSESSED = 49, 50
T_BLOCKED_PASS = 74

SHOT_TYPES = {T_MISS, T_POST, T_ATTEMPT_SAVED, T_GOAL}
DEFENSIVE_TYPES = {T_TACKLE: "Tackle", T_INTERCEPTION: "Interception", T_CLEARANCE: "Clearance"}
PRESSING_TYPES = {T_TACKLE: "Tackle", T_INTERCEPTION: "Interception", T_CHALLENGE: "Challenge"}

Q_LONG_BALL = 1
Q_CROSS, Q_THROUGH, Q_FREE_KICK, Q_CORNER = 2, 3, 5, 6
Q_HEAD = 15
Q_RIGHT_FOOT, Q_LEFT_FOOT = 20, 72
Q_END_X, Q_END_Y = 140, 141
Q_ZONE = 56
Q_REGULAR_PLAY, Q_FAST_BREAK, Q_SET_PIECE, Q_FROM_CORNER = 22, 23, 24, 25
Q_BIG_CHANCE = 80
Q_YELLOW_CARD, Q_SECOND_YELLOW, Q_RED_CARD = 31, 32, 33
Q_RELATED_EVENT = 233
Q_GOAL_KICK = 124
Q_CUTBACK = 195
Q_THROW_IN = 107

SET_PIECE_QIDS = {Q_FREE_KICK, Q_CORNER, Q_THROW_IN}
WIDE_THIRD = 100 / 3
SWITCH_MIN_DIST_M = 30
PITCH_X, PITCH_Y = 105.0, 68.0

ZONE14 = (70.0, 88.5, 27.2, 40.8)              # x0, x1, y0, y1
HALF_SPACES = [(52.5, 105.0, 13.6, 27.2), (52.5, 105.0, 40.8, 54.4)]
BOX_Y = (13.84, 54.16)

PPDA_ZONE_M = 63.0


def qmap(e):
    return {q["qualifierId"]: q.get("value") for q in e.get("qualifier", []) or []}


def has_q(e, qid):
    return any(q["qualifierId"] == qid for q in e.get("qualifier", []) or [])


def event_time(e):
    return e["timeMin"] * 60 + e["timeSec"]


def load_match(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    events = data["event"]
    events.sort(key=lambda e: (e["periodId"], event_time(e), e["eventId"]))
    return data["matchDetails"], events


def team_name(cid):
    # Falls back to ALL_TEAM_NAMES (all 16 teams, defined later in this module)
    # so TeamSnapshot.team_name resolves correctly for any team in the league
    # sample, not just this fixture's two sides in the small TEAM_NAMES dict.
    if cid in TEAM_NAMES:
        return TEAM_NAMES[cid]
    return ALL_TEAM_NAMES.get(cid, cid)


def team_short(cid):
    return TEAM_SHORT.get(cid, cid)


def to_m(x, y):
    return x * X_SCALE, y * Y_SCALE


def compute_attack_directions(events):
    """Disabled. This heuristic (flip a team's second-half coordinates if
    their average pass x sits above 50) was found to actively corrupt shot
    locations in the Hradec Kralove vs Besiktas report, and the sibling
    "Teplice vs Viktoria Plzen" report reproduced the same failure on two
    of the teams pooled into THIS report's own data (Plzeň's matchday-1 and
    matchday-2 games -- the same underlying event files, since Plzeň's
    season-to-date total is unchanged here). Re-running the diagnostic for
    this report's actual fixture teams (Hradec Kralove's 3 matches, Plzeň's
    2) with the flip disabled confirms every shot -- both teams, every
    match -- lands in a sensible attacking-third x range (roughly 75-105m
    of 105m); Hradec's own avg-pass-x-per-period never crosses 50 either
    way, so disabling costs Hradec nothing while fixing Plzeň's data.
    Returning {} makes norm_xy()'s directions.get(key, 1) default to
    no-flip for every event."""
    return {}


def norm_xy(e, directions):
    d = directions.get((e["contestantId"], e["periodId"]), 1)
    x, y = e["x"], e["y"]
    if d == 1:
        return x, y
    return 100.0 - x, 100.0 - y


def shot_angle_deg(x_m, y_m):
    dx = GOAL_X - x_m
    if dx <= 0:
        return 0.0
    y1 = y_m - (GOAL_Y - GOAL_WIDTH / 2)
    y2 = y_m - (GOAL_Y + GOAL_WIDTH / 2)
    denom = dx * dx + y1 * y2
    a = math.atan2(GOAL_WIDTH * dx, denom) if denom != 0 else math.pi / 2
    if a < 0:
        a += math.pi
    return math.degrees(a)


def shot_xg(x_m, y_m, is_header):
    dist = math.hypot(GOAL_X - x_m, GOAL_Y - y_m)
    angle = shot_angle_deg(x_m, y_m)
    z = -2.0 + 3.6 * math.radians(angle) - 0.085 * dist - (0.65 if is_header else 0.0)
    xg = 1.0 / (1.0 + math.exp(-z))
    return max(0.015, min(0.94, xg))


def xt_value(x100, y100):
    x_m, y_m = to_m(x100, y100)
    return shot_xg(x_m, y_m, is_header=False)


def build_shots(events, directions):
    rows = []
    for e in events:
        if e["typeId"] not in SHOT_TYPES or e.get("x") is None:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        is_header = has_q(e, Q_HEAD)
        qids = {q["qualifierId"] for q in e.get("qualifier", []) or []}
        dist = math.hypot(GOAL_X - xm, GOAL_Y - ym)
        angle = shot_angle_deg(xm, ym)
        xg = _XG_MODEL.score(dist, angle, ym, qids)
        outcome = {T_GOAL: "Goal", T_ATTEMPT_SAVED: "Saved", T_MISS: "Miss", T_POST: "Post"}[e["typeId"]]
        if has_q(e, Q_FROM_CORNER):
            situation = "Corner"
        elif has_q(e, Q_SET_PIECE):
            situation = "Set piece"
        elif has_q(e, Q_FAST_BREAK):
            situation = "Fast break"
        else:
            situation = "Open play"
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "eventId": e["eventId"],
            "minute": e["timeMin"],
            "period": e["periodId"],
            "x": xm, "y": ym,
            "outcome": outcome,
            "on_target": e["typeId"] in (T_GOAL, T_ATTEMPT_SAVED),
            "is_goal": e["typeId"] == T_GOAL,
            "is_header": is_header,
            "big_chance": has_q(e, Q_BIG_CHANCE),
            "situation": situation,
            "xg": xg,
        })
    return rows


def build_passes(events, directions):
    rows = []
    for e in events:
        if e["typeId"] != T_PASS or e.get("x") is None:
            continue
        q = qmap(e)
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        completed = e["outcome"] == 1
        end_x = end_y = ex = ey = None
        if Q_END_X in q and Q_END_Y in q:
            d = directions.get((e["contestantId"], e["periodId"]), 1)
            ex, ey = float(q[Q_END_X]), float(q[Q_END_Y])
            if d == -1:
                ex, ey = 100.0 - ex, 100.0 - ey
            end_x, end_y = to_m(ex, ey)
        start_dist = math.hypot(GOAL_X - xm, GOAL_Y - ym)
        end_dist = math.hypot(GOAL_X - end_x, GOAL_Y - end_y) if end_x is not None else None
        progressive = (completed and end_dist is not None and
                       end_dist <= start_dist * 0.75 and end_x > xm)
        xt_start = xt_value(x, y)
        xt_end = xt_value(ex, ey) if ex is not None else None
        xt_added = (xt_end - xt_start) if (completed and xt_end is not None) else 0.0

        is_switch = False
        if not (SET_PIECE_QIDS & set(q.keys())) and ex is not None:
            raw_y0, raw_y1 = e["y"], float(q[Q_END_Y])
            left0, right0 = raw_y0 <= WIDE_THIRD, raw_y0 >= (100 - WIDE_THIRD)
            left1, right1 = raw_y1 <= WIDE_THIRD, raw_y1 >= (100 - WIDE_THIRD)
            if (left0 and right1) or (right0 and left1):
                dx = (float(q[Q_END_X]) - e["x"]) / 100 * PITCH_X
                dy = (raw_y1 - raw_y0) / 100 * PITCH_Y
                if math.hypot(dx, dy) >= SWITCH_MIN_DIST_M:
                    is_switch = True

        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "playerId": e.get("playerId"),
            "minute": e["timeMin"], "second": e["timeSec"],
            "period": e["periodId"],
            "eventId": e["eventId"],
            "x": xm, "y": ym,
            "end_x": end_x, "end_y": end_y,
            "completed": completed,
            "is_cross": has_q(e, Q_CROSS),
            "is_corner": has_q(e, Q_CORNER),
            "is_long_ball": has_q(e, Q_LONG_BALL),
            "is_cutback": has_q(e, Q_CUTBACK),
            "is_goal_kick": has_q(e, Q_GOAL_KICK),
            "is_switch": is_switch and completed,
            "progressive": progressive,
            "final_third_entry": completed and start_dist > 35.0 and end_dist is not None and end_dist <= 35.0,
            "box_entry": (completed and end_x is not None and end_x >= 88.5
                          and 13.84 <= end_y <= 54.16 and not (xm >= 88.5 and 13.84 <= ym <= 54.16)),
            "xt_added": xt_added,
        })
    return rows


def build_defensive_actions(events, directions):
    rows = []
    for e in events:
        if e["typeId"] not in DEFENSIVE_TYPES or e.get("x") is None:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym,
            "action": DEFENSIVE_TYPES[e["typeId"]],
            "success": e.get("outcome", 1) == 1,
        })
    return rows


def build_pressing_actions(events, directions):
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        t = e["typeId"]
        if t in PRESSING_TYPES:
            action = PRESSING_TYPES[t]
        elif t == T_FOUL and e.get("outcome") == 0:
            action = "Foul"
        else:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym,
            "action": action,
        })
    return rows


def compute_ppda(passes, pressing_actions, contestant_id, lo=None, hi=None):
    """Opponent = "not contestant_id" rather than a single fixed id, so this
    still works when passes/pressing_actions are pooled across several
    matches (each match only ever has two sides, so "not us" == "whichever
    opponent we faced that game")."""
    def in_window(m):
        return (lo is None or m >= lo) and (hi is None or m < hi)

    opp_passes = sum(1 for p in passes if p["contestantId"] != contestant_id
                      and in_window(p["minute"]) and p["x"] <= PPDA_ZONE_M)
    def_actions = sum(1 for d in pressing_actions if d["contestantId"] == contestant_id
                       and in_window(d["minute"]) and d["x"] >= (105.0 - PPDA_ZONE_M))
    return opp_passes / def_actions if def_actions else float("nan")


def build_recoveries(events, directions):
    rows = []
    for e in events:
        if e["typeId"] != T_BALL_RECOVERY or e.get("x") is None:
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym,
        })
    return rows


def build_cards(events):
    rows = []
    for e in events:
        if e["typeId"] != T_CARD:
            continue
        if has_q(e, Q_RED_CARD) or has_q(e, Q_SECOND_YELLOW):
            kind = "Red" if has_q(e, Q_RED_CARD) else "2nd Yellow"
        else:
            kind = "Yellow"
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "minute": e["timeMin"],
            "kind": kind,
        })
    return rows


def build_touches(events, directions):
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        if e["typeId"] in (T_SUB_OFF, T_SUB_ON):
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "minute": e["timeMin"], "second": e["timeSec"], "period": e["periodId"],
            "x": xm, "y": ym,
            "typeId": e["typeId"],
        })
    return rows


def build_duels(events, directions):
    """Tackle + Aerial + Challenge, each contestantId's own outcome --
    same convention as the post-match report's _build_duels helper."""
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        if e["typeId"] not in (T_TACKLE, T_AERIAL, T_CHALLENGE):
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        rows.append({
            "contestantId": e["contestantId"],
            "team": team_name(e["contestantId"]),
            "player": e.get("playerName", "Unknown"),
            "minute": e["timeMin"], "period": e["periodId"],
            "x": xm, "y": ym,
            "action": {T_TACKLE: "Tackle", T_AERIAL: "Aerial", T_CHALLENGE: "Challenge"}[e["typeId"]],
            "success": e.get("outcome", 1) == 1,
        })
    return rows


def build_substitutions(events):
    rows = []
    for e in events:
        if e["typeId"] != T_SUB_OFF:
            continue
        rows.append({"contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
                      "player_off": e.get("playerName", "Unknown"), "minute": e["timeMin"]})
    return rows


def build_turnovers(events, directions):
    """A team's own failed pass or Dispossessed event in their own attacking
    half -- the moment they lost the ball already deep in the opponent's
    territory, handing it straight back in a dangerous area."""
    rows = []
    for e in events:
        if e.get("x") is None or not e.get("contestantId"):
            continue
        is_failed_pass = e["typeId"] == T_PASS and e.get("outcome") == 0
        is_dispossessed = e["typeId"] == T_DISPOSSESSED
        if not (is_failed_pass or is_dispossessed):
            continue
        x, y = norm_xy(e, directions)
        xm, ym = to_m(x, y)
        if xm < 52.5:
            continue
        rows.append({"contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
                      "player": e.get("playerName", "Unknown"), "minute": e["timeMin"], "period": e["periodId"],
                      "x": xm, "y": ym, "kind": "Failed pass" if is_failed_pass else "Dispossessed"})
    return rows


def build_shot_assists(events, directions, shots):
    """For each shot, the most recent completed pass by the shooting team
    since the ball last changed teams -- "who set this shot up"."""
    ball_events = [e for e in events if e.get("x") is not None and e.get("contestantId")
                   and (e["typeId"] in (T_PASS, T_TAKE_ON, T_TACKLE, T_INTERCEPTION, T_CLEARANCE,
                                        T_AERIAL, T_BALL_RECOVERY, T_DISPOSSESSED, 61)
                        or e["typeId"] in SHOT_TYPES)]
    ball_events.sort(key=lambda e: (e["periodId"], event_time(e), e["eventId"]))

    assists = {}
    current_team, pending_pass = None, None
    for e in ball_events:
        cid = e["contestantId"]
        if cid != current_team:
            current_team, pending_pass = cid, None
        if e["typeId"] in SHOT_TYPES:
            if pending_pass is not None and pending_pass.get("playerName") != e.get("playerName"):
                assists[e["eventId"]] = pending_pass
        elif e["typeId"] == T_PASS and e.get("outcome") == 1:
            pending_pass = e

    xg_by_eventid = {s["eventId"]: s["xg"] for s in shots}
    rows = []
    for e in events:
        if e["typeId"] not in SHOT_TYPES or e.get("x") is None:
            continue
        ap = assists.get(e["eventId"])
        if ap is None:
            continue
        x, y = norm_xy(ap, directions)
        xm, ym = to_m(x, y)
        q = qmap(ap)
        end_x = end_y = None
        if Q_END_X in q and Q_END_Y in q:
            d = directions.get((ap["contestantId"], ap["periodId"]), 1)
            ex, ey = float(q[Q_END_X]), float(q[Q_END_Y])
            if d == -1:
                ex, ey = 100.0 - ex, 100.0 - ey
            end_x, end_y = to_m(ex, ey)
        sx, sy = norm_xy(e, directions)
        sxm, sym = to_m(sx, sy)
        rows.append({
            "contestantId": e["contestantId"], "team": team_name(e["contestantId"]),
            "shooter": e.get("playerName", "Unknown"), "assister": ap.get("playerName", "Unknown"),
            "minute": e["timeMin"], "x": xm, "y": ym,
            "end_x": end_x if end_x is not None else sxm, "end_y": end_y if end_y is not None else sym,
            "shot_x": sxm, "shot_y": sym, "shot_xg": xg_by_eventid.get(e["eventId"], 0.0),
            "is_goal": e["typeId"] == T_GOAL,
        })
    return rows

### `TeamSnapshot` + Monte Carlo helper + league sample

`TeamSnapshot(team_id)` pools every match a team has played so far this season and exposes its shots, passes, defensive/pressing actions, touches, cards and duels, indexable by that team's own contestantId or "not that id" (numbers conceded).

In [3]:
class TeamSnapshot:
    """Everything derived from ALL of a team's played matches so far this
    season (season-to-date totals, not a single matchday snapshot): shots,
    passes, defensive/pressing actions, touches, cards etc, pooled across
    every match in ALL_TEAM_MATCHES[team_id] and indexable by either that
    team's own contestantId (its own numbers) or "not that id" (numbers
    conceded, whichever opponent was on the other side of each match).

    Per-match info (opponent, venue, own result) is kept in
    self.matches for anything that needs a per-match breakdown (e.g. a
    match log), while every aggregate stat below is summed/pooled/averaged
    across the full list."""

    def __init__(self, team_id):
        records = ALL_TEAM_MATCHES[team_id]
        self.team_id = team_id
        self.team_name = team_name(team_id)
        self.matches = []
        self.shots, self.passes, self.defs, self.pressing = [], [], [], []
        self.recoveries, self.cards, self.touches, self.duels = [], [], [], []
        self.subs, self.turnovers, self.assists = [], [], []

        for idx, m in enumerate(records):
            is_home = m["home_id"] == team_id
            opponent_id = m["away_id"] if is_home else m["home_id"]
            opponent_name = m["away_name"] if is_home else m["home_name"]
            match_details, events = load_match(m["path"])
            directions = compute_attack_directions(events)
            shots = build_shots(events, directions)
            for s in shots:
                s["match_idx"] = idx  # which entry in self.matches this shot's own build-up chain lives in

            scores = match_details["scores"]["ft"]
            own_score = scores["home"] if is_home else scores["away"]
            opp_score = scores["away"] if is_home else scores["home"]
            outcome = "Won" if own_score > opp_score else ("Lost" if own_score < opp_score else "Drew")
            venue_desc = "at home" if is_home else "away"
            result = f"{outcome} {own_score}-{opp_score} {venue_desc}"

            self.matches.append(dict(
                matchday=m["matchday"], opponent_id=opponent_id, opponent_name=opponent_name,
                was_home=is_home, outcome=outcome, own_score=own_score, opp_score=opp_score,
                result=result, events=events, directions=directions,
            ))
            self.shots += shots
            self.passes += build_passes(events, directions)
            self.defs += build_defensive_actions(events, directions)
            self.pressing += build_pressing_actions(events, directions)
            self.recoveries += build_recoveries(events, directions)
            self.cards += build_cards(events)
            self.touches += build_touches(events, directions)
            self.duels += build_duels(events, directions)
            self.subs += build_substitutions(events)
            self.turnovers += build_turnovers(events, directions)
            self.assists += build_shot_assists(events, directions, shots)

        self.matches_played = len(self.matches)
        self.wins = sum(1 for m in self.matches if m["outcome"] == "Won")
        self.draws = sum(1 for m in self.matches if m["outcome"] == "Drew")
        self.losses = sum(1 for m in self.matches if m["outcome"] == "Lost")
        self.points = self.wins * 3 + self.draws
        self.record = f"{self.wins}W {self.draws}D {self.losses}L"

    def own(self, rows):
        return [r for r in rows if r["contestantId"] == self.team_id]

    def against(self, rows):
        return [r for r in rows if r["contestantId"] != self.team_id]

    @property
    def opponents_label(self):
        """A single opponent's name if only one match is on record, else a
        generic plural -- used in dek/caption text that used to say "vs
        {single opponent}" when every page was built from one match."""
        if self.matches_played == 1:
            return self.matches[0]["opponent_name"]
        return "their opponents"

    @property
    def xg_for(self):
        return sum(s["xg"] for s in self.own(self.shots))

    @property
    def xg_against(self):
        return sum(s["xg"] for s in self.against(self.shots))

    def ppda_for(self):
        return compute_ppda(self.passes, self.pressing, self.team_id)

    def verticality(self):
        fwd = [p["end_x"] - p["x"] for p in self.own(self.passes)
               if p["completed"] and p["end_x"] is not None and p["end_x"] > p["x"]]
        return sum(fwd) / len(fwd) if fwd else 0.0

    def def_line_height(self):
        d = self.own(self.defs) + [p for p in self.own(self.pressing) if p["action"] != "Foul"]
        return sum(p["x"] for p in d) / len(d) if d else float("nan")

    def halfspace_zone14_counts(self):
        passes = [p for p in self.own(self.passes) if p["completed"] and p["end_x"] is not None]
        zone14 = sum(1 for p in passes if ZONE14[0] <= p["end_x"] < ZONE14[1] and ZONE14[2] <= p["end_y"] < ZONE14[3])
        halfspace = sum(1 for p in passes if any(x0 <= p["end_x"] < x1 and y0 <= p["end_y"] < y1
                                                  for x0, x1, y0, y1 in HALF_SPACES))
        return halfspace, zone14

    def touch_share(self):
        h = len(self.own(self.touches))
        a = len(self.against(self.touches))
        return h / (h + a) if (h + a) else float("nan")

    def field_tilt(self):
        h = sum(1 for t in self.own(self.touches) if t["x"] >= 70)
        a = sum(1 for t in self.against(self.touches) if t["x"] >= 70)
        return h / (h + a) if (h + a) else float("nan")

    def pass_tempo_mps(self):
        """Avg metres/second of this team's completed passes -- pass
        distance divided by the time gap to the next event in that same
        match (any team). Gaps > 8s (stoppages, fouls, VAR checks) are
        excluded as noise, not genuine tempo. Computed per match (event
        order only means anything within one match) then pooled across
        every match played."""
        speeds = []
        for m in self.matches:
            events, directions = m["events"], m["directions"]
            ordered = sorted(events, key=lambda e: (e["periodId"], event_time(e), e["eventId"]))
            for i in range(len(ordered) - 1):
                e = ordered[i]
                if e["typeId"] != T_PASS or e.get("contestantId") != self.team_id or e.get("outcome") != 1:
                    continue
                nxt = ordered[i + 1]
                if nxt["periodId"] != e["periodId"]:
                    continue
                dt = event_time(nxt) - event_time(e)
                if dt <= 0 or dt > 8:
                    continue
                q = qmap(e)
                if Q_END_X not in q or Q_END_Y not in q:
                    continue
                x, y = norm_xy(e, directions)
                xm, ym = to_m(x, y)
                d = directions.get((e["contestantId"], e["periodId"]), 1)
                ex, ey = float(q[Q_END_X]), float(q[Q_END_Y])
                if d == -1:
                    ex, ey = 100.0 - ex, 100.0 - ey
                exm, eym = to_m(ex, ey)
                dist = math.hypot(exm - xm, eym - ym)
                speeds.append(dist / dt)
        return sum(speeds) / len(speeds) if speeds else 0.0

    def pass_volume(self):
        return len(self.own(self.passes))


def simulate_scorelines(home_shots, away_shots, n=20000, seed=42, cap=6):
    """Monte Carlo simulation cross-pairing each fixture side's own MW1
    shot-xG list (their shot volume + quality from the one match on
    record) as a form proxy for this unplayed fixture. Same mechanic as
    the post-match report's simulate_scorelines, but the two shot lists
    come from two different, unrelated matches -- an early-season style
    projection, not a model fit to this specific pairing."""
    import random
    rng = random.Random(seed)
    home_xgs = [s["xg"] for s in home_shots]
    away_xgs = [s["xg"] for s in away_shots]

    score_counts = {}
    home_goals, away_goals = [], []
    for _ in range(n):
        h = sum(1 for xg in home_xgs if rng.random() < xg)
        a = sum(1 for xg in away_xgs if rng.random() < xg)
        home_goals.append(h)
        away_goals.append(a)
        key = (min(h, cap), min(a, cap))
        score_counts[key] = score_counts.get(key, 0) + 1

    home_win = sum(1 for h, a in zip(home_goals, away_goals) if h > a) / n
    draw = sum(1 for h, a in zip(home_goals, away_goals) if h == a) / n
    away_win = sum(1 for h, a in zip(home_goals, away_goals) if h < a) / n
    return {
        "score_counts": score_counts, "n": n, "cap": cap,
        "home_win": home_win, "draw": draw, "away_win": away_win,
        "home_goal_dist": home_goals, "away_goal_dist": away_goals,
    }


# ---------------------------------------------------------------------------
# Full 16-team league sample -- every match feed this repo has for the
# season so far (matchdays 1-2, 16 matches, all 16 teams), used by the
# pace-vs-volume and verticality-ranking pages and by TeamSnapshot itself
# for this fixture's two sides. Season to date, not a single round -- once
# more matchdays are played, adding their event files to ALL_MATCHES is
# the only change needed for every page to pick them up automatically,
# since TeamSnapshot pools over however many matches ALL_TEAM_MATCHES
# lists for a given team.
# ---------------------------------------------------------------------------

ALL_MATCHES = [
    # Matchday 1 (2026-07-25 to 2026-07-27)
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-25_FC Viktoria Plzeň - FC Slovan Liberec.json"),
         home_id="c6fx1460nlkawjgh67sp7a1hd", home_name="Viktoria Plzeň",
         away_id="2c4rs2vp0tyjiqa7y3gfttf24", away_name="Slovan Liberec"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-25_FC Zbrojovka Brno - AC Sparta Praha.json"),
         home_id="6k350zwynsc23f0sxw9akgc6y", home_name="Zbrojovka Brno",
         away_id="5ocdn3a6s75u0d0dy0rbou0xc", away_name="Sparta Praha"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-25_FC Zlín - FC Baník Ostrava.json"),
         home_id=ZLIN_ID, home_name="Zlín", away_id=OSTRAVA_ID, away_name="Baník Ostrava"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-25_FK Teplice - Bohemians Praha 1905.json"),
         home_id=TEPLICE_ID, home_name="Teplice", away_id=BOHEMIANS_ID, away_name="Bohemians 1905"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-26_FC Hradec Králové - FK Pardubice.json"),
         home_id=HRADEC_ID, home_name="Hradec Králové", away_id=PARDUBICE_ID, away_name="Pardubice"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-26_FK Jablonec - SK Sigma Olomouc.json"),
         home_id="bdz8tx20ekj1ryi2e2u13jdl", home_name="Jablonec",
         away_id="dchxm00ei80l8ljbcfpdill8k", away_name="Sigma Olomouc"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-26_SK Slavia Praha - 1. FC Slovácko.json"),
         home_id="8kpapuorr6hf0vosnovbreqqd", home_name="Slavia Praha",
         away_id="bp5x8iw8pstucx6s4iqht6xqf", away_name="Slovácko"),
    dict(matchday=1, path=os.path.join(EVENTS_DIR, "2026-07-27_SK Artis Brno - FK Mladá Boleslav.json"),
         home_id=ARTIS_BRNO_ID, home_name="Artis Brno", away_id=MLADA_BOLESLAV_ID, away_name="Mladá Boleslav"),
    # Matchday 2 (2026-07-31 to 2026-08-02)
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-07-31_AC Sparta Praha - FC Zlín.json"),
         home_id="5ocdn3a6s75u0d0dy0rbou0xc", home_name="Sparta Praha", away_id=ZLIN_ID, away_name="Zlín"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-01_1. FC Slovácko - SK Artis Brno.json"),
         home_id="bp5x8iw8pstucx6s4iqht6xqf", home_name="Slovácko", away_id=ARTIS_BRNO_ID, away_name="Artis Brno"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-01_FC Baník Ostrava - SK Slavia Praha.json"),
         home_id=OSTRAVA_ID, home_name="Baník Ostrava",
         away_id="8kpapuorr6hf0vosnovbreqqd", away_name="Slavia Praha"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-01_FC Slovan Liberec - FK Teplice.json"),
         home_id="2c4rs2vp0tyjiqa7y3gfttf24", home_name="Slovan Liberec", away_id=TEPLICE_ID, away_name="Teplice"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-01_FC Viktoria Plzeň - FC Zbrojovka Brno.json"),
         home_id="c6fx1460nlkawjgh67sp7a1hd", home_name="Viktoria Plzeň",
         away_id="6k350zwynsc23f0sxw9akgc6y", away_name="Zbrojovka Brno"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-02_Bohemians Praha 1905 - FC Hradec Králové.json"),
         home_id=BOHEMIANS_ID, home_name="Bohemians 1905", away_id=HRADEC_ID, away_name="Hradec Králové"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-02_FK Pardubice - FK Jablonec.json"),
         home_id=PARDUBICE_ID, home_name="Pardubice", away_id="bdz8tx20ekj1ryi2e2u13jdl", away_name="Jablonec"),
    dict(matchday=2, path=os.path.join(EVENTS_DIR, "2026-08-02_SK Sigma Olomouc - FK Mladá Boleslav.json"),
         home_id="dchxm00ei80l8ljbcfpdill8k", home_name="Sigma Olomouc",
         away_id=MLADA_BOLESLAV_ID, away_name="Mladá Boleslav"),
    # Matchday 3 -- only Hradec Kralove's game has been played + has an event
    # feed so far (2026-08-09); every other team in this table is still on
    # its matchday-1 + matchday-2 totals.
    dict(matchday=3, path=os.path.join(EVENTS_DIR, "2026-08-09_FC Hradec Králové - FC Baník Ostrava.json"),
         home_id=HRADEC_ID, home_name="Hradec Králové", away_id=OSTRAVA_ID, away_name="Baník Ostrava"),
]

ALL_TEAM_NAMES = {}
ALL_TEAM_MATCHES = {}
for m in ALL_MATCHES:
    ALL_TEAM_NAMES[m["home_id"]] = m["home_name"]
    ALL_TEAM_NAMES[m["away_id"]] = m["away_name"]
    ALL_TEAM_MATCHES.setdefault(m["home_id"], []).append(m)
    ALL_TEAM_MATCHES.setdefault(m["away_id"], []).append(m)


class LeagueSeason:
    """Every team with a match feed in this repo, one season-to-date
    TeamSnapshot each (pools however many matches ALL_TEAM_MATCHES lists
    for that team -- 2 each right now, matchdays 1-2)."""

    def __init__(self):
        self.teams = {tid: TeamSnapshot(tid) for tid in ALL_TEAM_NAMES}

    def ranking(self, metric_fn, reverse=True):
        vals = [(tid, metric_fn(tm)) for tid, tm in self.teams.items()]
        vals = [(tid, v) for tid, v in vals if v == v]
        return sorted(vals, key=lambda kv: kv[1], reverse=reverse)

### Quick sanity check

Same checks run at the bottom of `match_data.py` when executed directly: each team's record/xG/PPDA, plus the attack-direction diagnostic (avg raw pass x per team per period) that motivated disabling `compute_attack_directions()` above.

In [4]:
tep = TeamSnapshot(HRADEC_ID)
plz = TeamSnapshot(PLZEN_ID)
for snap in (tep, plz):
    print(team_name(snap.team_id), f"({snap.matches_played} matches played):", snap.record,
          f"({snap.points} pts)")
    for m in snap.matches:
        print(f"    MD{m['matchday']}: {m['result']} vs {m['opponent_name']}")
    print("  xG for/against:", round(snap.xg_for, 2), round(snap.xg_against, 2))
    print("  shots for/against:", len(snap.own(snap.shots)), len(snap.against(snap.shots)))
    print("  PPDA:", round(snap.ppda_for(), 2))

print("\nAttack-direction sanity check (avg raw pass x per team per period, both teams' own matches):")
for snap in (tep, plz):
    for idx, m in enumerate(snap.matches):
        events = m["events"]
        sums = {}
        for e in events:
            if e["typeId"] != T_PASS or e.get("x") is None:
                continue
            if e["x"] == 0 and e["y"] == 0:
                continue
            key = (e["contestantId"], e["periodId"])
            s = sums.setdefault(key, [0.0, 0])
            s[0] += e["x"]
            s[1] += 1
        for (cid, period), (total, n) in sorted(sums.items()):
            avg = total / n if n else float("nan")
            flag = "  <-- near 50 threshold" if abs(avg - 50) < 5 else ""
            print(f"    MD{m['matchday']} {team_short(cid)} P{period}: avg pass x = {avg:.1f}{flag}")

FC Hradec Králové (3 matches played): 2W 1D 0L (7 pts)
    MD1: Won 2-1 at home vs Pardubice
    MD2: Drew 0-0 away vs Bohemians 1905
    MD3: Won 2-1 at home vs Baník Ostrava
  xG for/against: 4.38 2.14
  shots for/against: 35 38
  PPDA: 15.94
FC Viktoria Plzeň (2 matches played): 0W 1D 1L (1 pts)
    MD1: Lost 1-3 at home vs Slovan Liberec
    MD2: Drew 1-1 at home vs Zbrojovka Brno
  xG for/against: 3.27 2.66
  shots for/against: 22 27
  PPDA: 16.1

Attack-direction sanity check (avg raw pass x per team per period, both teams' own matches):
    MD1 Hradec Kr. P1: avg pass x = 47.3  <-- near 50 threshold
    MD1 Hradec Kr. P2: avg pass x = 45.3  <-- near 50 threshold
    MD1 Pardubice P1: avg pass x = 46.6  <-- near 50 threshold
    MD1 Pardubice P2: avg pass x = 44.9
    MD2 Hradec Kr. P1: avg pass x = 43.5
    MD2 Hradec Kr. P2: avg pass x = 48.4  <-- near 50 threshold
    MD2 Bohemians P1: avg pass x = 45.8  <-- near 50 threshold
    MD2 Bohemians P2: avg pass x = 55.3
    MD3 Hra

## Chart building (`build_charts.py`)

All 50 chart functions, Meridian house style (dark). Split into cells at each numbered section boundary from the source file.

In [5]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from mplsoccer import Pitch

sys.path.insert(0, REPO_ROOT)
from housestyle import style, components
from housestyle.colors import CATEGORICAL_DARK, STATUS_DARK

# This notebook defines match_data.py's contents directly above, in the
# same kernel namespace, so "import match_data as md" would re-run the
# module from disk (reloading the xG model, re-parsing every match) --
# alias the current module (this notebook) as md instead, same objects.
import sys as _sys
md = _sys.modules["__main__"]

OUT_DIR = os.path.join(NOTEBOOK_DIR, "Visuals")
os.makedirs(OUT_DIR, exist_ok=True)

FIGSIZE = (13.33, 7.5)
HKR_C = CATEGORICAL_DARK[2]    # teal -- FC Hradec Králové (home)
PLZ_C = CATEGORICAL_DARK[0]    # ink blue -- FC Viktoria Plzeň (away)
GOOD_C = STATUS_DARK["good"]
WARN_C = STATUS_DARK["warning"]
LEAGUE_MUTED = "#5A6672"      # league-context gray, distinct from house axis gray
HKR_SHORT, PLZ_SHORT = "Hradec Kr.", "Plzeň"

BOX_Y = (13.84, 54.16)


def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=170, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("Saved:", path)


def new_fig():
    palette, cats = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    return fig, palette


def new_pitch(palette):
    return Pitch(pitch_type="uefa", pitch_color=palette["surface"], line_color=palette["axis"],
                 linewidth=1.0, half=False, line_zorder=2, pad_left=2, pad_right=2)


def team_color(cid):
    if cid == md.HRADEC_ID:
        return HKR_C
    if cid == md.PLZEN_ID:
        return PLZ_C
    return None  # a non-fixture opponent -- caller supplies a muted color


def team_short(cid):
    return md.TEAM_SHORT.get(cid, cid)

In [6]:
# ---------------------------------------------------------------------------
# 01. Cover
# ---------------------------------------------------------------------------

def cover():
    palette, _ = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(palette["surface"])

    fig.text(0.5, 0.64, md.FIXTURE_HOME_NAME.upper(), fontsize=34, fontweight="bold",
              color=HKR_C, family="serif", ha="center", va="center")
    fig.text(0.5, 0.555, "vs", fontsize=16, color=palette["ink_muted"],
              family="sans-serif", ha="center", va="center")
    fig.text(0.5, 0.47, md.FIXTURE_AWAY_NAME.upper(), fontsize=34, fontweight="bold",
              color=PLZ_C, family="serif", ha="center", va="center")

    fig.text(0.5, 0.375, f"Kickoff {md.KICKOFF_LOCAL}, {md.MATCH_DATE}", fontsize=14,
              fontweight="bold", color=palette["ink_primary"], family="sans-serif",
              ha="center", va="center")

    fig.text(0.5, 0.30, f"{md.COMPETITION}  ·  {md.VENUE}", fontsize=12,
              color=palette["ink_secondary"], family="sans-serif", ha="center", va="center")

    fig.text(0.5, 0.19, f"{components.MARK} PRE-MATCH PREVIEW  ·  50 PAGES", fontsize=13, fontweight="bold",
              color=palette["accent"], family="sans-serif", ha="center", va="center")
    fig.text(0.5, 0.145, "Built from each team's matches played so far this season -- early-season form, not head-to-head",
              fontsize=9.5, color=palette["ink_muted"], family="sans-serif", ha="center", va="center")

    components.brand_mark(fig, palette=palette, right=0.94, y=0.93)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "01_cover.png")


# ---------------------------------------------------------------------------
# 02. Fixture context
# ---------------------------------------------------------------------------

def fixture_context(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.06, 0.14, 0.88, 0.58])
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    facts = [
        ("Competition", md.COMPETITION),
        ("Venue", f"{md.VENUE} (Hradec Králové home)"),
        ("Kickoff", f"{md.MATCH_DATE}, {md.KICKOFF_LOCAL} local"),
        ("Round", "Matchday 5 -- season-to-date totals below reflect each side's matches played so far"),
    ]
    y0 = 1.0
    for i, (label, val) in enumerate(facts):
        y = y0 - i * 0.075
        ax.text(0.0, y, label.upper(), fontsize=9.5, fontweight="bold", color=palette["accent"], va="top")
        ax.text(0.30, y, val, fontsize=11.5, color=palette["ink_primary"], va="top")

    ax.axhline(0.66, xmin=0, xmax=1, color=palette["axis"], linewidth=1.0)

    col_x = [0.0, 0.52]
    for x, snap, name, color in zip(col_x, (hkr, plz), (md.FIXTURE_HOME_NAME, md.FIXTURE_AWAY_NAME), (HKR_C, PLZ_C)):
        y = 0.58
        ax.text(x, y, f"{name}  ({snap.record}, {snap.points} pts)", fontsize=14, fontweight="bold",
                color=color, va="top", family="serif")
        y -= 0.075
        for m in snap.matches:
            ax.text(x, y, f"MD{m['matchday']}: {m['result']} vs {m['opponent_name']}", fontsize=10.5,
                    color=palette["ink_primary"], va="top")
            y -= 0.06
        y -= 0.015
        ax.text(x, y, f"xG created / conceded: {snap.xg_for:.2f} / {snap.xg_against:.2f}", fontsize=10.5,
                color=palette["ink_secondary"], va="top")
        y -= 0.06
        ax.text(x, y, f"Shots for / against: {len(snap.own(snap.shots))} / {len(snap.against(snap.shots))}",
                fontsize=10.5, color=palette["ink_secondary"], va="top")
        y -= 0.06
        ax.text(x, y, f"PPDA (pressing intensity): {snap.ppda_for():.1f}", fontsize=10.5,
                color=palette["ink_secondary"], va="top")

    fig.text(0.06, 0.155, "Every page in this preview is built from all of that team's matches played so far "
                          "this season, pooled together -- an early-season style snapshot (3 games for Hradec, "
                          "2 for Plzeň), not a head-to-head (these two sides have not yet met this season).",
              fontsize=9, color=palette["ink_muted"], ha="left", va="top", wrap=True)

    components.header(fig, kicker="Fixture Preview",
                       title=f"{md.FIXTURE_HOME_NAME} host {md.FIXTURE_AWAY_NAME}",
                       dek="What each side has shown across their matches so far this season",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "02_fixture_context.png")


# ---------------------------------------------------------------------------
# 03. Matchday-1 shot maps, one pitch per team (for + against)
# ---------------------------------------------------------------------------

def shot_maps_mw1(hkr, plz):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.02, 0.10, 0.47, 0.62])
    ax2 = fig.add_axes([0.51, 0.10, 0.47, 0.62])
    pitch.draw(ax=ax1)
    pitch.draw(ax=ax2)

    for ax, snap, color, name in ((ax1, hkr, HKR_C, md.FIXTURE_HOME_NAME), (ax2, plz, PLZ_C, md.FIXTURE_AWAY_NAME)):
        for s in snap.shots:
            is_own = s["contestantId"] == snap.team_id
            c = color if is_own else palette["ink_muted"]
            size = 70 + s["xg"] * 800
            if s["is_goal"]:
                pitch.scatter(s["x"], s["y"], ax=ax, s=size, marker="o", color=c,
                              edgecolors=palette["ink_primary"], linewidth=1.4, zorder=5)
            else:
                pitch.scatter(s["x"], s["y"], ax=ax, s=size, marker="o", facecolors="none",
                              edgecolors=c, linewidth=1.4, alpha=0.8, zorder=4)
        ax.set_title(f"{name}\n{snap.record} across {snap.matches_played} matches", color=color, fontsize=12,
                     fontweight="bold", family="sans-serif")

    legend_elems = [Line2D([0], [0], marker="o", color=palette["surface"], markerfacecolor=palette["ink_muted"],
                            markeredgecolor=palette["ink_muted"], markersize=10, label="Shot conceded", linewidth=0),
                    Line2D([0], [0], marker="o", color=palette["surface"], markerfacecolor=palette["surface"],
                            markeredgecolor=palette["ink_primary"], markersize=10, label="Own shot / goal", linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Season So Far",
                       title="Shot maps from every match played this season, own goal on the left",
                       dek="Filled = goal  ·  Size = xG  ·  Own team's own colour, muted = conceded  ·  all matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "03_shot_maps_mw1.png")


# ---------------------------------------------------------------------------
# 04. xG snapshot comparison
# ---------------------------------------------------------------------------

def xg_snapshot(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.24, 0.16, 0.66, 0.56])

    metrics = [
        ("xG created", hkr.xg_for, plz.xg_for),
        ("xG conceded", hkr.xg_against, plz.xg_against),
        ("Shots", len(hkr.own(hkr.shots)), len(plz.own(plz.shots))),
        ("Big chances", sum(1 for s in hkr.own(hkr.shots) if s["big_chance"]),
         sum(1 for s in plz.own(plz.shots) if s["big_chance"])),
    ]
    n = len(metrics)
    ypos = np.arange(n)[::-1]
    maxval = max(max(b, h) for _, b, h in metrics) * 1.25 or 1
    for y, (label, b, h) in zip(ypos, metrics):
        ax.barh(y + 0.18, b, height=0.32, color=HKR_C)
        ax.barh(y - 0.18, h, height=0.32, color=PLZ_C)
        fb = f"{b:.2f}" if isinstance(b, float) else str(b)
        fh = f"{h:.2f}" if isinstance(h, float) else str(h)
        ax.text(b + maxval * 0.02, y + 0.18, fb, va="center", fontsize=10, color=palette["ink_primary"])
        ax.text(h + maxval * 0.02, y - 0.18, fh, va="center", fontsize=10, color=palette["ink_primary"])
    ax.set_yticks(ypos)
    ax.set_yticklabels([m[0] for m in metrics], fontsize=11.5, color=palette["ink_primary"])
    ax.set_xlim(0, maxval)
    ax.grid(axis="x")
    ax.set_axisbelow(True)

    legend_elems = [Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=HKR_C,
                            markersize=12, label=md.FIXTURE_HOME_NAME, linewidth=0),
                    Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=PLZ_C,
                            markersize=12, label=md.FIXTURE_AWAY_NAME, linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Chance Quality",
                       title="Hradec have created more and conceded a bit less, so far this season",
                       dek=f"Season-to-date shot numbers, trained xG model  ·  {hkr.matches_played} matches "
                           f"for Hradec, {plz.matches_played} for Plzeň",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "04_xg_snapshot.png")

In [7]:
# ---------------------------------------------------------------------------
# 05-06. Pass network -- one page per team
# ---------------------------------------------------------------------------

def _average_positions(team_passes, min_passes=6):
    completed = [p for p in team_passes if p["completed"] and p["end_x"] is not None]
    by_player = {}
    for p in completed:
        by_player.setdefault(p["player"], []).append((p["x"], p["y"]))
    return {pl: (np.mean([v[0] for v in vs]), np.mean([v[1] for v in vs]), len(vs))
            for pl, vs in by_player.items() if len(vs) >= min_passes}


def _combinations(team_passes, avg_pos):
    combos = {}
    ordered = sorted(team_passes, key=lambda p: (p["period"], p["minute"] * 60 + p["second"]))
    for i in range(len(ordered) - 1):
        p, nxt = ordered[i], ordered[i + 1]
        if not p["completed"]:
            continue
        if p["player"] not in avg_pos or nxt["player"] not in avg_pos or p["player"] == nxt["player"]:
            continue
        key = tuple(sorted((p["player"], nxt["player"])))
        combos[key] = combos.get(key, 0) + 1
    return combos


def _draw_pass_network(ax, team_passes, color, palette, pitch, min_passes=6, node_scale=1.15, label_size=8.6):
    avg_pos = _average_positions(team_passes, min_passes)
    combos = _combinations(team_passes, avg_pos)
    pitch.draw(ax=ax)
    max_c = max(combos.values()) if combos else 1
    for (p1, p2), c in combos.items():
        if c < 2:
            continue
        x1, y1, _ = avg_pos[p1]
        x2, y2, _ = avg_pos[p2]
        pitch.lines(x1, y1, x2, y2, ax=ax, color=color, alpha=0.25 + 0.5 * (c / max_c),
                    lw=0.6 + 3.0 * (c / max_c), zorder=2)
    max_n = max(v[2] for v in avg_pos.values()) if avg_pos else 1
    for pl, (x, y, n) in avg_pos.items():
        size = (260 + 900 * (n / max_n)) * node_scale
        pitch.scatter(x, y, ax=ax, s=size, color=palette["surface"], edgecolors=color,
                      linewidth=2.0, zorder=4)
        last = pl.split(" ")[-1]
        pitch.annotate(last, (x, y), ax=ax, ha="center", va="center", fontsize=label_size,
                       color=palette["ink_primary"], fontweight="bold", zorder=5)
    return avg_pos, combos


def pass_network_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    team_passes = snap.own(snap.passes)

    ax = fig.add_axes([0.02, 0.10, 0.54, 0.62])
    _draw_pass_network(ax, team_passes, color, palette, pitch)

    by_player = {}
    for p in team_passes:
        by_player.setdefault(p["player"], {"att": 0, "comp": 0, "prog": 0, "box": 0, "cross": 0})
        d = by_player[p["player"]]
        d["att"] += 1
        d["comp"] += int(p["completed"])
        d["prog"] += int(p["progressive"])
        d["box"] += int(p["box_entry"])
        d["cross"] += int(p["is_cross"] and p["completed"])

    ax2 = fig.add_axes([0.60, 0.14, 0.37, 0.58])
    ax2.axis("off")
    rows = sorted(by_player.items(), key=lambda kv: -kv[1]["att"])[:14]
    headers = ["Player", "Pass", "Acc%", "Prog", "Box", "Cross"]
    col_x = [0.0, 0.46, 0.58, 0.72, 0.84, 0.94]
    for x, h in zip(col_x, headers):
        ax2.text(x, 1.0, h, fontsize=9.5, fontweight="bold", color=palette["ink_primary"], va="top",
                 ha="left" if x == 0 else "center")
    ax2.axhline(0.975, color=palette["axis"], linewidth=0.9)
    row_h = 0.94 / max(len(rows), 1)
    for i, (pl, d) in enumerate(rows):
        y = 0.94 - i * row_h
        acc = d["comp"] / d["att"] if d["att"] else 0
        vals = [pl, str(d["att"]), f"{acc:.0%}", str(d["prog"]), str(d["box"]), str(d["cross"])]
        for x, v in zip(col_x, vals):
            ax2.text(x, y, v, fontsize=8.8, color=palette["ink_secondary"], va="top",
                     ha="left" if x == 0 else "center")
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1.03)

    components.header(fig, kicker="Pass Network",
                       title=f"{name}: how they've built play so far this season",
                       dek=f"Average completed-pass position (≥ 6 passes), {snap.matches_played} matches pooled, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_pass_network_{slug}.png")


# ---------------------------------------------------------------------------
# 07-08. Touch heatmap -- one page per team
# ---------------------------------------------------------------------------

def touch_heatmap_team_page(snap, color, name, page_num, slug, cmap):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.06, 0.10, 0.88, 0.62])
    pitch.draw(ax=ax)

    own = snap.own(snap.touches)
    xs = [t["x"] for t in own]
    ys = [t["y"] for t in own]
    stats = pitch.bin_statistic(xs, ys, statistic="count", bins=(9, 6))
    pitch.heatmap(stats, ax=ax, cmap=cmap, edgecolors=palette["surface"], alpha=0.92, zorder=1)

    components.header(fig, kicker="Territory",
                       title=f"{name}: where they've spent their {len(xs)} touches so far this season",
                       dek=f"Touch density by pitch zone, {snap.matches_played} matches pooled, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_touch_heatmap_{slug}.png")


# ---------------------------------------------------------------------------
# 09. Possession thirds
# ---------------------------------------------------------------------------

def possession_thirds(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.16, 0.24, 0.68, 0.40])

    zone_colors = [CATEGORICAL_DARK[0], CATEGORICAL_DARK[3], CATEGORICAL_DARK[1]]
    zone_labels = ["Defensive", "Middle", "Attacking"]

    def thirds(snap):
        t = snap.own(snap.touches)
        d = sum(1 for x in t if x["x"] < 35)
        m = sum(1 for x in t if 35 <= x["x"] < 70)
        a = sum(1 for x in t if x["x"] >= 70)
        total = d + m + a
        return [d / total, m / total, a / total], total

    for i, (snap, name, color) in enumerate(((hkr, md.FIXTURE_HOME_NAME, HKR_C), (plz, md.FIXTURE_AWAY_NAME, PLZ_C))):
        fracs, total = thirds(snap)
        y = 1 - i
        left = 0
        for frac, zc, zl in zip(fracs, zone_colors, zone_labels):
            ax.barh(y, frac, left=left, height=0.6, color=zc)
            if frac > 0.06:
                ax.text(left + frac / 2, y, f"{frac:.0%}", ha="center", va="center",
                        fontsize=10.5, fontweight="bold", color=palette["surface"])
            left += frac
        ax.text(-0.02, y, f"{name}\n({total} touches)", ha="right", va="center", fontsize=10.5,
                fontweight="bold", color=color)

    ax.set_xlim(0, 1)
    ax.set_ylim(-0.7, 1.7)
    ax.set_yticks([])
    ax.set_xlabel("Share of touches")
    ax.set_axisbelow(True)

    legend_elems = [Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=zc,
                            markersize=12, label=zl, linewidth=0) for zc, zl in zip(zone_colors, zone_labels)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 0.06), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Possession",
                       title="Each side's territory split so far this season",
                       dek="Distribution of touches across pitch thirds, own attacking direction, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "19_possession_thirds.png")


# ---------------------------------------------------------------------------
# 10. Progression comparison bars
# ---------------------------------------------------------------------------

def progression_bars(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.24, 0.16, 0.66, 0.56])

    bp = hkr.own(hkr.passes)
    hp = plz.own(plz.passes)
    metrics = [
        ("Progressive passes", sum(1 for p in bp if p["progressive"]), sum(1 for p in hp if p["progressive"])),
        ("Final-third entries", sum(1 for p in bp if p["final_third_entry"]),
         sum(1 for p in hp if p["final_third_entry"])),
        ("Passes into the box", sum(1 for p in bp if p["box_entry"]), sum(1 for p in hp if p["box_entry"])),
        ("Completed crosses", sum(1 for p in bp if p["is_cross"] and p["completed"]),
         sum(1 for p in hp if p["is_cross"] and p["completed"])),
    ]
    n = len(metrics)
    ypos = np.arange(n)[::-1]
    maxval = max(max(b, h) for _, b, h in metrics) * 1.15
    for y, (label, b, h) in zip(ypos, metrics):
        ax.barh(y + 0.18, b, height=0.32, color=HKR_C)
        ax.barh(y - 0.18, h, height=0.32, color=PLZ_C)
        ax.text(b + maxval * 0.015, y + 0.18, str(b), va="center", fontsize=10, color=palette["ink_primary"])
        ax.text(h + maxval * 0.015, y - 0.18, str(h), va="center", fontsize=10, color=palette["ink_primary"])
    ax.set_yticks(ypos)
    ax.set_yticklabels([m[0] for m in metrics], fontsize=11.5, color=palette["ink_primary"])
    ax.set_xlim(0, maxval)
    ax.set_xlabel("Count")
    ax.grid(axis="x")
    ax.set_axisbelow(True)

    legend_elems = [Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=HKR_C,
                            markersize=12, label=md.FIXTURE_HOME_NAME, linewidth=0),
                    Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=PLZ_C,
                            markersize=12, label=md.FIXTURE_AWAY_NAME, linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Progression",
                       title="How each side has moved the ball forward so far this season",
                       dek="Progressive pass = completed pass cutting ≥25% off the distance to goal, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "20_progression_bars.png")

In [8]:
# ---------------------------------------------------------------------------
# 11. PPDA / pressing comparison
# ---------------------------------------------------------------------------

def ppda_pressing(hkr, plz):
    fig, palette = new_fig()
    ax1 = fig.add_axes([0.08, 0.16, 0.40, 0.58])
    ax2 = fig.add_axes([0.56, 0.16, 0.40, 0.58])

    buckets = [(0, 15), (15, 30), (30, 45), (45, 60), (60, 75), (75, 96)]
    labels = ["0-15", "15-30", "30-45", "45-60", "60-75", "75-90+"]

    def bucketed(snap):
        return [md.compute_ppda(snap.passes, snap.pressing, snap.team_id, lo, hi)
                for lo, hi in buckets]

    for ax, snap, color, name in ((ax1, hkr, HKR_C, md.FIXTURE_HOME_NAME), (ax2, plz, PLZ_C, md.FIXTURE_AWAY_NAME)):
        vals = bucketed(snap)
        finite = [v for v in vals if not math.isnan(v)]
        top = max(finite) * 1.15 if finite else 1.0
        xs = np.arange(len(labels))
        clean = [v if not math.isnan(v) else 0 for v in vals]
        ax.bar(xs, clean, color=color)
        for x, v in zip(xs, vals):
            if not math.isnan(v):
                ax.text(x, v + top * 0.02, f"{v:.1f}", ha="center", fontsize=9.5,
                        color=palette["ink_primary"], fontweight="bold")
        overall = snap.ppda_for()
        ax.axhline(overall, color=palette["ink_muted"], linestyle="--", linewidth=1.0)
        ax.set_xticks(xs)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylim(0, top)
        ax.set_title(f"{name}\nOverall PPDA: {overall:.1f}", color=color, fontsize=11.5,
                     fontweight="bold", family="sans-serif")
        ax.set_ylabel("PPDA")

    components.header(fig, kicker="Pressing",
                       title="Pressing intensity so far this season, by 15-minute window",
                       dek="Passes per defensive action in the opponent's own 60%  ·  lower = more intense press  ·  "
                           "windows pooled across matches played",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "23_ppda_pressing.png")


# ---------------------------------------------------------------------------
# 12. Defensive actions
# ---------------------------------------------------------------------------

def defensive_actions(hkr, plz):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.02, 0.10, 0.47, 0.62])
    ax2 = fig.add_axes([0.51, 0.10, 0.47, 0.62])
    pitch.draw(ax=ax1)
    pitch.draw(ax=ax2)

    markers = {"Tackle": "o", "Interception": "D", "Clearance": "s"}
    action_colors = {"Tackle": CATEGORICAL_DARK[2], "Interception": CATEGORICAL_DARK[3],
                      "Clearance": palette["ink_muted"]}

    for ax, snap, color, name in ((ax1, hkr, HKR_C, md.FIXTURE_HOME_NAME), (ax2, plz, PLZ_C, md.FIXTURE_AWAY_NAME)):
        team_defs = snap.own(snap.defs)
        for action, marker in markers.items():
            pts = [d for d in team_defs if d["action"] == action]
            if not pts:
                continue
            xs = [p["x"] for p in pts]
            ys = [p["y"] for p in pts]
            pitch.scatter(xs, ys, ax=ax, s=80, marker=marker, color=action_colors[action],
                          edgecolors=palette["surface"], linewidth=0.6, alpha=0.9, zorder=4)
        counts = {a: sum(1 for d in team_defs if d["action"] == a) for a in markers}
        title = f"{name}\nTkl {counts['Tackle']}  ·  Int {counts['Interception']}  ·  Clr {counts['Clearance']}"
        ax.set_title(title, color=color, fontsize=12, fontweight="bold", family="sans-serif")

    legend_elems = [Line2D([0], [0], marker=markers[a], color=palette["surface"], markerfacecolor=action_colors[a],
                            markersize=10, label=a, linewidth=0) for a in markers]
    fig.legend(handles=legend_elems, loc="lower center", ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Defending",
                       title="Where each side has won the ball back so far this season",
                       dek="Tackles, interceptions and clearances, own goal on the left, attacking right, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "24_defensive_actions.png")


# ---------------------------------------------------------------------------
# 13. Duels & discipline
# ---------------------------------------------------------------------------

def duels_discipline(hkr, plz):
    fig, palette = new_fig()
    ax1 = fig.add_axes([0.08, 0.16, 0.55, 0.58])
    ax2 = fig.add_axes([0.72, 0.16, 0.24, 0.58])

    kinds = ["Tackle", "Aerial", "Challenge"]
    n = len(kinds)
    ypos = np.arange(n)[::-1]
    maxval = 0
    rows = []
    for kind in kinds:
        b = [d for d in hkr.own(hkr.duels) if d["action"] == kind]
        h = [d for d in plz.own(plz.duels) if d["action"] == kind]
        b_won = sum(1 for d in b if d["success"])
        h_won = sum(1 for d in h if d["success"])
        rows.append((kind, len(b), b_won, len(h), h_won))
        maxval = max(maxval, len(b), len(h))
    maxval *= 1.25

    for y, (kind, b_n, b_won, h_n, h_won) in zip(ypos, rows):
        b_rate = b_won / b_n if b_n else 0
        h_rate = h_won / h_n if h_n else 0
        ax1.barh(y + 0.18, b_n, height=0.32, color=palette["axis"])
        ax1.barh(y + 0.18, b_won, height=0.32, color=HKR_C)
        ax1.barh(y - 0.18, h_n, height=0.32, color=palette["axis"])
        ax1.barh(y - 0.18, h_won, height=0.32, color=PLZ_C)
        ax1.text(b_n + maxval * 0.015, y + 0.18, f"{b_won}/{b_n} ({b_rate:.0%})", va="center",
                fontsize=9, color=palette["ink_primary"])
        ax1.text(h_n + maxval * 0.015, y - 0.18, f"{h_won}/{h_n} ({h_rate:.0%})", va="center",
                fontsize=9, color=palette["ink_primary"])
    ax1.set_yticks(ypos)
    ax1.set_yticklabels([f"{k} duels" for k in kinds], fontsize=11, color=palette["ink_primary"])
    ax1.set_xlim(0, maxval)
    ax1.set_xlabel("Contested (solid = won)")
    ax1.grid(axis="x")
    ax1.set_axisbelow(True)
    ax1.set_title("Duel win rates", color=palette["ink_primary"], fontsize=11.5, fontweight="bold",
                   family="sans-serif")

    def fouls_cards(snap):
        fouls = sum(1 for d in snap.own(snap.pressing) if d["action"] == "Foul")
        yellow = sum(1 for c in snap.own(snap.cards) if c["kind"] == "Yellow")
        red = sum(1 for c in snap.own(snap.cards) if c["kind"] in ("Red", "2nd Yellow"))
        return fouls, yellow, red

    b_fouls, b_yellow, b_red = fouls_cards(hkr)
    h_fouls, h_yellow, h_red = fouls_cards(plz)
    disc_metrics = [("Fouls", b_fouls, h_fouls), ("Yellows", b_yellow, h_yellow), ("Reds", b_red, h_red)]
    n2 = len(disc_metrics)
    ypos2 = np.arange(n2)[::-1]
    maxval2 = max(max(b, h) for _, b, h in disc_metrics) * 1.4 or 1
    for y, (label, b, h) in zip(ypos2, disc_metrics):
        ax2.barh(y + 0.18, b, height=0.32, color=HKR_C)
        ax2.barh(y - 0.18, h, height=0.32, color=PLZ_C)
        ax2.text(b + maxval2 * 0.03, y + 0.18, str(b), va="center", fontsize=9, color=palette["ink_primary"])
        ax2.text(h + maxval2 * 0.03, y - 0.18, str(h), va="center", fontsize=9, color=palette["ink_primary"])
    ax2.set_yticks(ypos2)
    ax2.set_yticklabels([m[0] for m in disc_metrics], fontsize=10, color=palette["ink_primary"])
    ax2.set_xlim(0, maxval2)
    ax2.set_title("Discipline", color=palette["ink_primary"], fontsize=11.5, fontweight="bold", family="sans-serif")

    legend_elems = [Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=HKR_C,
                            markersize=12, label=md.FIXTURE_HOME_NAME, linewidth=0),
                    Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=PLZ_C,
                            markersize=12, label=md.FIXTURE_AWAY_NAME, linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Physicality",
                       title="Duels contested and discipline so far this season",
                       dek="Tackle, aerial and loose-ball duels, plus fouls and cards, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "25_duels_discipline.png")


# ---------------------------------------------------------------------------
# 14-15. Key players to watch -- one page per team
# ---------------------------------------------------------------------------

def key_players_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.10, 0.16, 0.80, 0.56])

    def score_players():
        scores = {}
        for s in snap.own(snap.shots):
            scores[s["player"]] = scores.get(s["player"], 0) + s["xg"] + (3.0 if s["is_goal"] else 0)
        for p in snap.own(snap.passes):
            scores[p["player"]] = scores.get(p["player"], 0) + 0.15 * p["progressive"] + 0.35 * p["box_entry"]
        for d in snap.own(snap.defs):
            scores[d["player"]] = scores.get(d["player"], 0) + 0.3
        return sorted(scores.items(), key=lambda kv: -kv[1])[:8]

    top = score_players()[::-1]
    ypos = np.arange(len(top))
    vals = [v for _, v in top]
    ax.barh(ypos, vals, color=color)
    ax.set_yticks(ypos)
    ax.set_yticklabels([p for p, _ in top], fontsize=11, color=palette["ink_primary"])
    for y, v in zip(ypos, vals):
        ax.text(v + max(vals, default=1) * 0.02, y, f"{v:.1f}", va="center", fontsize=9.5,
                color=palette["ink_secondary"])
    ax.set_xlabel("Impact score (season to date)")

    fig.text(0.5, 0.10, "Simple composite: xG + 3×goals + 0.15×progressive pass + 0.35×box entry + "
                         f"0.3×defensive action  ·  not an official rating, {snap.matches_played} matches of evidence",
              ha="center", fontsize=8.8, color=palette["ink_muted"])

    components.header(fig, kicker="Players To Watch",
                       title=f"{name}: who has stood out so far this season",
                       dek=f"{snap.matches_played} matches pooled  ·  shooting, progression and defending combined",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_key_players_{slug}.png")

In [9]:
# ---------------------------------------------------------------------------
# 16. Team radar
# ---------------------------------------------------------------------------

def team_radar(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.26, 0.18, 0.48, 0.58], polar=True)

    bp = hkr.own(hkr.passes)
    hp = plz.own(plz.passes)
    b_touch = sum(1 for t in hkr.own(hkr.touches) if t["x"] >= 70)
    h_touch = sum(1 for t in plz.own(plz.touches) if t["x"] >= 70)

    metrics = [
        ("xG created", hkr.xg_for, plz.xg_for),
        ("xG conceded (inv.)", 1 / max(hkr.xg_against, 0.05), 1 / max(plz.xg_against, 0.05)),
        ("Shots", len(hkr.own(hkr.shots)), len(plz.own(plz.shots))),
        ("Progressive passes", sum(1 for p in bp if p["progressive"]), sum(1 for p in hp if p["progressive"])),
        ("Box entries", sum(1 for p in bp if p["box_entry"]), sum(1 for p in hp if p["box_entry"])),
        ("Final-third touches", b_touch, h_touch),
        ("Pass accuracy", sum(1 for p in bp if p["completed"]) / len(bp), sum(1 for p in hp if p["completed"]) / len(hp)),
        ("Pressing (inv. PPDA)", 1 / hkr.ppda_for(), 1 / plz.ppda_for()),
    ]
    labels = [m[0] for m in metrics]
    n = len(labels)
    boh_norm = [m[1] / max(m[1], m[2], 1e-9) for m in metrics]
    hkr_norm = [m[2] / max(m[1], m[2], 1e-9) for m in metrics]

    angles = [i / n * 2 * math.pi for i in range(n)] + [0]
    for vals, color, name in ((boh_norm, HKR_C, md.FIXTURE_HOME_NAME), (hkr_norm, PLZ_C, md.FIXTURE_AWAY_NAME)):
        pts = vals + [vals[0]]
        ax.plot(angles, pts, color=color, linewidth=2.2, marker="o", markersize=4, label=name, zorder=3)
        ax.fill(angles, pts, color=color, alpha=0.15, zorder=2)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=8.8, color=palette["ink_primary"])
    ax.set_yticks([])
    ax.set_ylim(0, 1.15)
    ax.spines["polar"].set_color(palette["axis"])
    ax.grid(color=palette["grid"])

    fig.legend(loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.065),
               fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Shape Comparison",
                       title="Season-to-date numbers side by side",
                       dek="Each axis normalized to the better of the two teams that metric (=1.0)  ·  "
                           "different opponents, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "45_team_radar.png")


# ---------------------------------------------------------------------------
# 17. Keys to the game (narrative)
# ---------------------------------------------------------------------------

def keys_to_the_game(hkr, plz):
    palette, _ = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(palette["surface"])

    points = []
    if plz.xg_against < hkr.xg_against:
        points.append(f"Plzeň have conceded less ({plz.xg_against:.2f} xG) than Hradec have "
                       f"({hkr.xg_against:.2f} xG) across their {plz.matches_played} matches so far.")
    else:
        points.append(f"Hradec have conceded a bit less ({hkr.xg_against:.2f} xG across {hkr.matches_played} "
                       f"matches) than Plzeň have ({plz.xg_against:.2f} xG) -- most of that gap came in "
                       f"Plzeň's 1-3 home defeat to Slovan Liberec, so their defensive record is shakier than "
                       "the single draw with Zbrojovka Brno suggests.")
    if abs(hkr.ppda_for() - plz.ppda_for()) < 1.0:
        points.append(f"Pressing intensity is essentially a wash so far (PPDA {hkr.ppda_for():.1f} for Hradec "
                       f"vs {plz.ppda_for():.1f} for Plzeň) -- neither side has shown a clear high-press "
                       "identity yet, so expect this to be decided elsewhere.")
    elif hkr.ppda_for() < plz.ppda_for():
        points.append(f"Hradec have pressed higher so far (PPDA {hkr.ppda_for():.1f} vs "
                       f"{plz.ppda_for():.1f}) -- expect them to try to disrupt Plzeň's build-up early "
                       f"rather than sit off.")
    else:
        points.append(f"Plzeň have pressed considerably higher so far (PPDA {plz.ppda_for():.1f} vs "
                       f"Hradec's {hkr.ppda_for():.1f}) -- if that intensity travels on the road, Hradec's "
                       f"buildup play will be under pressure from the first whistle.")
    b_touch = sum(1 for t in hkr.own(hkr.touches) if t["x"] >= 70)
    h_touch = sum(1 for t in plz.own(plz.touches) if t["x"] >= 70)
    if h_touch > b_touch:
        points.append(f"Plzeň have actually racked up more final-third touches ({h_touch} vs "
                       f"Hradec's {b_touch}) but converted them far less efficiently ({plz.xg_for:.2f} xG from "
                       f"{len(plz.own(plz.shots))} shots vs Hradec's {hkr.xg_for:.2f} from "
                       f"{len(hkr.own(hkr.shots))}) -- territory alone hasn't been enough for them without "
                       "sharper shot selection.")
    else:
        points.append(f"Hradec have spent more of their matches in the final third ({b_touch} touches there "
                       f"vs Plzeň's {h_touch}) -- if that territorial edge holds at home, Plzeň will need "
                       "to defend deep for longer spells.")
    points.append(f"This reads as a real but not overwhelming gap: Hradec are unbeaten so far "
                   f"({hkr.record}, {hkr.points} pts, {hkr.matches_played} matches played) and have the better "
                   f"underlying numbers, while Plzeň ({plz.record}, {plz.points} pts) are still searching for "
                   "their first win, having followed their opening defeat with a home draw against Zbrojovka "
                   "Brno -- but Hradec's edge in xG for/against is moderate, not one-sided, so this is not a "
                   "guaranteed home banker.")

    fig.text(0.5, 0.075, f"★ Small-sample caveat: every number above is drawn from {hkr.matches_played} matches "
                          f"for Hradec and {plz.matches_played} for Plzeň so far this season. Treat as an early "
                          "style signal, not settled form.",
              ha="center", fontsize=9, color=palette["ink_muted"], style="italic")

    ax = fig.add_axes([0.08, 0.16, 0.84, 0.58])
    ax.axis("off")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    row_h = 1.0 / len(points)
    for i, txt in enumerate(points):
        y = 1.0 - (i + 0.15) * row_h
        ax.text(0.0, y, f"{i + 1}.", fontsize=15, fontweight="bold", color=palette["accent"], va="top")
        ax.text(0.06, y, txt, fontsize=12, color=palette["ink_primary"], va="top", wrap=True,
                linespacing=1.5)

    components.header(fig, kicker="Keys To The Game",
                       title="Four things worth watching for on 2026-08-23",
                       dek="Reasoned from each side's season-to-date numbers",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "48_keys_to_the_game.png")


# ---------------------------------------------------------------------------
# 18. Win probability (Monte Carlo, cross-paired season-to-date shot lists)
# ---------------------------------------------------------------------------

def _donut(ax, frac, color, palette, label, sublabel):
    ax.pie([frac, 1 - frac], radius=1.0, startangle=90, counterclock=False,
           colors=[color, palette["axis"]], wedgeprops=dict(width=0.32, edgecolor=palette["surface"], linewidth=1.5))
    ax.text(0, 0.12, f"{frac:.0%}", ha="center", va="center", fontsize=20, fontweight="bold", color=palette["ink_primary"])
    ax.text(0, -0.12, sublabel, ha="center", va="center", fontsize=8.5, color=palette["ink_muted"])
    ax.set_title(label, color=color, fontsize=11.5, fontweight="bold", family="sans-serif", pad=2)


def win_probability(hkr, plz, sim):
    fig, palette = new_fig()
    ax1 = fig.add_axes([0.06, 0.34, 0.19, 0.32])
    ax2 = fig.add_axes([0.28, 0.34, 0.19, 0.32])
    _donut(ax1, sim["home_win"], HKR_C, palette, HKR_SHORT, "WIN PROBABILITY")
    _donut(ax2, sim["away_win"], PLZ_C, palette, PLZ_SHORT, "WIN PROBABILITY")
    fig.text(0.275, 0.30, f"Draw: {sim['draw']:.0%}", ha="center", fontsize=10.5,
              color=palette["ink_secondary"], fontweight="bold")

    ax3 = fig.add_axes([0.56, 0.20, 0.38, 0.52])
    cap = sim["cap"]
    n = sim["n"]
    grid = {}
    for (h, a), c in sim["score_counts"].items():
        grid[(h, a)] = c / n
    top_scores = sorted(grid.items(), key=lambda kv: -kv[1])[:6]
    labels = [f"{h}-{a}" if h < cap and a < cap else f"{h}+{'-' if h>=cap else ''}{a}{'+' if a>=cap else ''}"
              for (h, a), _ in top_scores]
    vals = [v for _, v in top_scores]
    ypos = np.arange(len(vals))[::-1]
    ax3.barh(ypos, vals, color=palette["accent"])
    ax3.set_yticks(ypos)
    ax3.set_yticklabels(labels, fontsize=10)
    for y, v in zip(ypos, vals):
        ax3.text(v + max(vals) * 0.02, y, f"{v:.1%}", va="center", fontsize=9.5, color=palette["ink_primary"])
    ax3.set_xlim(0, max(vals) * 1.25)
    ax3.set_xlabel("Simulated probability")
    ax3.set_title(f"Most likely scorelines ({HKR_SHORT}–{PLZ_SHORT})", color=palette["ink_primary"],
                   fontsize=11.5, fontweight="bold", family="sans-serif")
    ax3.grid(axis="x")
    ax3.set_axisbelow(True)

    components.header(fig, kicker="Match Projection",
                       title=(f"{md.FIXTURE_HOME_NAME} rate as favourites ({sim['home_win']:.0%})"
                              if sim["home_win"] >= sim["away_win"]
                              else f"{md.FIXTURE_AWAY_NAME} rate as favourites ({sim['away_win']:.0%})"),
                       dek=f"{n:,}-simulation Monte Carlo cross-pairing each side's season-to-date shot xG list "
                           "-- an early-season projection, not a fitted model",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "49_win_probability.png")


# ---------------------------------------------------------------------------
# 19. Report card (closing summary)
# ---------------------------------------------------------------------------

def report_card(hkr, plz, sim):
    palette, _ = style.apply("dark")
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(palette["surface"])

    fig.text(0.5, 0.90, f"{md.FIXTURE_HOME_NAME}  vs  {md.FIXTURE_AWAY_NAME}",
              fontsize=18, fontweight="bold", color=palette["ink_primary"], family="serif",
              ha="center", va="center")
    fig.text(0.5, 0.855, f"{md.COMPETITION}  ·  {md.VENUE}  ·  {md.MATCH_DATE}, {md.KICKOFF_LOCAL} local",
              fontsize=10.5, color=palette["ink_secondary"], ha="center", va="center")

    bp = hkr.own(hkr.passes)
    hp = plz.own(plz.passes)
    rows = [
        ("Season record", f"{hkr.record} ({hkr.points} pts)", f"{plz.record} ({plz.points} pts)"),
        ("xG created", f"{hkr.xg_for:.2f}", f"{plz.xg_for:.2f}"),
        ("xG conceded", f"{hkr.xg_against:.2f}", f"{plz.xg_against:.2f}"),
        ("Pass accuracy", f"{sum(1 for p in bp if p['completed']) / len(bp):.0%}",
         f"{sum(1 for p in hp if p['completed']) / len(hp):.0%}"),
        ("PPDA", f"{hkr.ppda_for():.1f}", f"{plz.ppda_for():.1f}"),
        ("Projected win prob.", f"{sim['home_win']:.0%}", f"{sim['away_win']:.0%}"),
    ]

    ax = fig.add_axes([0.14, 0.20, 0.72, 0.55])
    ax.axis("off")
    ax.text(0.0, 1.0, md.FIXTURE_HOME_NAME, fontsize=12.5, fontweight="bold", color=HKR_C, ha="left", va="top")
    ax.text(1.0, 1.0, md.FIXTURE_AWAY_NAME, fontsize=12.5, fontweight="bold", color=PLZ_C, ha="right", va="top")
    n = len(rows)
    for i, (label, bval, hval) in enumerate(rows):
        y = 0.85 - i * (0.85 / n)
        ax.text(0.0, y, bval, fontsize=13, fontweight="bold", color=palette["ink_primary"], ha="left", va="top")
        ax.text(0.5, y, label, fontsize=10.5, color=palette["ink_muted"], ha="center", va="top")
        ax.text(1.0, y, hval, fontsize=13, fontweight="bold", color=palette["ink_primary"], ha="right", va="top")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    components.brand_mark(fig, palette=palette, right=0.94, y=0.965)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "50_report_card.png")

In [10]:
# ---------------------------------------------------------------------------
# 05-06. xG flow -- one page per team, every match played so far
# ---------------------------------------------------------------------------

def xg_flow_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.08, 0.16, 0.78, 0.60])

    def series(rows):
        team_shots = sorted(rows, key=lambda s: s["minute"])
        mins, cum, total = [0.0], [0.0], 0.0
        for s in team_shots:
            mins.append(s["minute"]); cum.append(total)
            total += s["xg"]
            mins.append(s["minute"]); cum.append(total)
        mins.append(96); cum.append(total)
        return mins, cum

    own_shots = snap.own(snap.shots)
    opp_shots = snap.against(snap.shots)
    for rows, color_, label in ((own_shots, color, name), (opp_shots, palette["ink_muted"], snap.opponents_label)):
        mins, cum = series(rows)
        ax.plot(mins, cum, color=color_, linewidth=2.4, zorder=4)
        ax.fill_between(mins, cum, step=None, color=color_, alpha=0.10, zorder=1)
        ax.annotate(f"{label}\n{cum[-1]:.2f} xG", xy=(1, cum[-1]), xycoords=("axes fraction", "data"),
                    xytext=(10, 0), textcoords="offset points", color=color_, fontsize=10,
                    fontweight="bold", va="center", ha="left", annotation_clip=False)

    for rows, color_ in ((own_shots, color), (opp_shots, palette["ink_muted"])):
        running = 0.0
        for s in sorted(rows, key=lambda s: s["minute"]):
            if s["is_goal"]:
                ax.scatter([s["minute"]], [running], marker="*", s=200, color=palette["ink_primary"],
                           edgecolors=color_, linewidth=1.6, zorder=6)
            running += s["xg"]

    ax.axvline(45, color=palette["axis"], linewidth=0.8, linestyle=":")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Minute of match")
    ax.set_ylabel("Cumulative xG")

    components.header(fig, kicker="xG Flow",
                       title=f"{name}: cumulative xG so far this season ({snap.record})",
                       dek=f"{snap.matches_played} matches pooled by minute-of-match  ·  own xG model, this team's coloured, opponents muted",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_xg_flow_{slug}.png")


# ---------------------------------------------------------------------------
# 07-08. Shot quality table -- one page per team, every match played so far
# ---------------------------------------------------------------------------

def shot_quality_table_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()

    both = snap.own(snap.shots) + snap.against(snap.shots)
    ordered = sorted(both, key=lambda s: (snap.matches[s["match_idx"]]["matchday"], s["minute"]))
    cols = ["MD", "Min", "Team", "Player", "Situation", "Body", "Outcome", "xG"]
    widths = [0.06, 0.08, 0.20, 0.24, 0.16, 0.12, 0.09, 0.05]

    # Season totals run to 2+ matches' worth of shots (vs. a single match), so
    # split into two side-by-side columns once the list gets too long for one
    # column to stay legible at a fixed font size.
    n_cols = 2 if len(ordered) > 22 else 1
    col_bounds = [0.03, 0.51] if n_cols == 2 else [0.05]
    col_w = 0.46 if n_cols == 2 else 0.90
    per_col = math.ceil(len(ordered) / n_cols)
    chunks = [ordered[i:i + per_col] for i in range(0, len(ordered), per_col)] or [[]]

    for col_x0, chunk in zip(col_bounds, chunks):
        ax = fig.add_axes([col_x0, 0.12, col_w, 0.62])
        ax.axis("off")
        ax.set_xlim(0, 1)
        x0 = [sum(widths[:i]) for i in range(len(widths))]

        header_y = 1.0
        for x, w, label in zip(x0, widths, cols):
            ax.text(x, header_y, label, fontsize=9.5, fontweight="bold", color=palette["ink_primary"],
                    va="top", ha="left")
        ax.axhline(header_y - 0.025, xmin=0, xmax=1, color=palette["axis"], linewidth=1.0)

        row_h = 0.95 / max(per_col, 1)
        for i, s in enumerate(chunk):
            y = header_y - 0.05 - i * row_h
            is_own = s["contestantId"] == snap.team_id
            c = color if is_own else palette["ink_muted"]
            weight = "bold" if s["is_goal"] else "normal"
            match = snap.matches[s["match_idx"]]
            team_label = name.split(" ")[-1] if is_own else match["opponent_name"].split(" ")[-1]
            outcome_label = f"★ {s['outcome']}" if s["is_goal"] else s["outcome"]
            vals = [str(match["matchday"]), f"{s['minute']}'", team_label,
                    s["player"], s["situation"], "Head" if s["is_header"] else "Foot", outcome_label, f"{s['xg']:.2f}"]
            fsize = 8.5 if n_cols == 2 else 9.5
            for x, w, v in zip(x0, widths, vals):
                col_color = GOOD_C if (x == x0[6] and s["is_goal"]) else (c if x == x0[2] else palette["ink_primary"])
                ax.text(x, y, v, fontsize=fsize, color=col_color,
                        fontweight=weight, va="top", ha="left")
        ax.set_ylim(header_y - 0.05 - per_col * row_h, 1.03)

    components.header(fig, kicker="Shot Log",
                       title=f"{name}: all {len(ordered)} shots across their {snap.matches_played} matches this season",
                       dek="own xG model: distance + angle to goal, header penalty applied  ·  MD = matchday",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_shot_quality_table_{slug}.png")


# ---------------------------------------------------------------------------
# 09-10. Goal build-ups -- one page per team, every match played so far
# ---------------------------------------------------------------------------

def goal_buildups_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)

    both = snap.own(snap.shots) + snap.against(snap.shots)
    goals = sorted([s for s in both if s["is_goal"]],
                   key=lambda s: (snap.matches[s["match_idx"]]["matchday"], s["minute"]))
    n = max(len(goals), 1)
    axes = [fig.add_axes([0.02 + i * (0.96 / n), 0.10, 0.96 / n - 0.02, 0.62]) for i in range(len(goals))]

    for ax, g in zip(axes, goals):
        pitch.draw(ax=ax)
        is_own = g["contestantId"] == snap.team_id
        c = color if is_own else palette["ink_muted"]
        match = snap.matches[g["match_idx"]]
        match_events, match_directions = match["events"], match["directions"]
        team_events = [e for e in match_events if e["contestantId"] == g["contestantId"]
                       and e.get("x") is not None and e["typeId"] in (1, 3, 61)
                       and md.event_time(e) <= g["minute"] * 60 + 59]
        team_events.sort(key=lambda e: (e["periodId"], md.event_time(e), e["eventId"]))
        chain = team_events[-4:]
        pts = []
        for e in chain:
            x, y = md.norm_xy(e, match_directions)
            xm, ym = md.to_m(x, y)
            pts.append((xm, ym))
        pts.append((g["x"], g["y"]))

        for j in range(len(pts) - 1):
            x1, y1 = pts[j]
            x2, y2 = pts[j + 1]
            alpha = 0.45 + 0.55 * (j / (len(pts) - 1))
            pitch.arrows(x1, y1, x2, y2, ax=ax, color=c, alpha=alpha, width=2.2,
                        headwidth=6, headlength=6, zorder=3)
        pitch.scatter(g["x"], g["y"], ax=ax, s=260, marker="*", color=palette["ink_primary"],
                      edgecolors=c, linewidth=1.6, zorder=6)
        team_label = name if is_own else match["opponent_name"]
        ax.set_title(f"MD{match['matchday']} {g['minute']}'  {g['player']}\n{team_label}", color=c,
                     fontsize=10.5, fontweight="bold", family="sans-serif")

    fig.text(0.5, 0.085, "Last 4 touches before each goal, both teams, across this team's matches played so far",
              ha="center", fontsize=9, color=palette["ink_muted"])

    components.header(fig, kicker="Goal Build-Ups",
                       title=f"{name}: how all {len(goals)} goals so far this season were made",
                       dek=f"{snap.record} across {snap.matches_played} matches",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_goal_buildups_{slug}.png")


# ---------------------------------------------------------------------------
# 13-14. Progressive passes -- one page per team
# ---------------------------------------------------------------------------

def progressive_passes_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    team_passes = snap.own(snap.passes)
    prog = [p for p in team_passes if p["progressive"]]

    ax = fig.add_axes([0.02, 0.10, 0.96, 0.62])
    pitch.draw(ax=ax)
    for p in prog:
        is_box = p["box_entry"]
        pitch.arrows(p["x"], p["y"], p["end_x"], p["end_y"], ax=ax,
                    color=GOOD_C if is_box else color, alpha=0.9 if is_box else 0.55,
                    width=2.4 if is_box else 1.4, headwidth=6, headlength=6,
                    zorder=4 if is_box else 3)

    legend_elems = [Line2D([0], [0], color=color, lw=2.0, label="Progressive pass"),
                    Line2D([0], [0], color=GOOD_C, lw=2.4, label="...into the box")]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.06), fontsize=10.5, labelcolor=palette["ink_secondary"])

    n_box = sum(1 for p in prog if p["box_entry"])
    components.header(fig, kicker="Progression",
                       title=f"{name}: {len(prog)} progressive passes, {n_box} of them straight into the box",
                       dek="Progressive pass = completed pass cutting ≥25% off the distance to goal, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_progressive_passes_{slug}.png")

In [11]:
# ---------------------------------------------------------------------------
# 15-16. Passing directness -- one page per team
# ---------------------------------------------------------------------------

def passing_directness_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.10, 0.16, 0.82, 0.58])

    by_player = {}
    for p in snap.own(snap.passes):
        d = by_player.setdefault(p["player"], {"gain": 0.0, "att": 0, "comp": 0})
        d["att"] += 1
        d["comp"] += int(p["completed"])
        if p["completed"] and p["end_x"] is not None:
            d["gain"] += p["end_x"] - p["x"]

    items = [(pl, d) for pl, d in by_player.items() if d["att"] >= 5]
    xs = [d["gain"] for _, d in items]
    ys = [d["comp"] / d["att"] for _, d in items]
    sizes = [40 + d["att"] * 6 for _, d in items]
    ax.scatter(xs, ys, s=sizes, color=color, alpha=0.85, edgecolors=palette["surface"], linewidth=0.8, zorder=3)
    for (pl, d), x, y in zip(items, xs, ys):
        ax.annotate(pl.split(" ")[-1], xy=(x, y), xytext=(6, 4), textcoords="offset points",
                    fontsize=8.5, color=palette["ink_secondary"])
    ax.axhline(np.mean(ys) if ys else 0, color=palette["axis"], linewidth=0.8, linestyle="--")
    ax.set_xlabel("Net metres gained by completed passes (forward - backward)")
    ax.set_ylabel("Pass completion %")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

    components.header(fig, kicker="Passing Profile",
                       title=f"{name}: who progressed the ball, and how safely",
                       dek="Players with ≥ 5 pass attempts  ·  bubble size = passes attempted",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_passing_directness_{slug}.png")


# ---------------------------------------------------------------------------
# 21-22. Field tilt over time -- one page per team, every match played so far
# ---------------------------------------------------------------------------

def field_tilt_over_time_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.16, 0.16, 0.78, 0.58])

    bucket = 5
    max_min = 95
    edges = list(range(0, max_min + bucket, bucket))
    tilt, centers = [], []
    own_t = snap.own(snap.touches)
    opp_t = snap.against(snap.touches)
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        o = sum(1 for t in own_t if lo <= t["minute"] < hi and t["x"] >= 70)
        a = sum(1 for t in opp_t if lo <= t["minute"] < hi and t["x"] >= 70)
        total = o + a
        tilt.append((o / total - 0.5) * 100 if total else 0.0)
        centers.append((lo + hi) / 2)

    tilt = np.array(tilt)
    centers = np.array(centers)
    ax.fill_between(centers, tilt, 0, where=(tilt >= 0), color=color, alpha=0.75, step="mid")
    ax.fill_between(centers, tilt, 0, where=(tilt < 0), color=palette["ink_muted"], alpha=0.75, step="mid")
    ax.axhline(0, color=palette["axis"], linewidth=1.0)
    ax.axvline(45, color=palette["axis"], linewidth=0.8, linestyle=":")

    ax.set_ylim(-55, 55)
    ax.set_xlim(0, max_min)
    ax.set_xlabel("Minute")
    ax.set_ylabel("Field tilt (final-third touch share)")
    ax.set_yticks([-50, -25, 0, 25, 50])
    ax.set_yticklabels(["Opponents 100%", "75%", "Even", "75%", f"{name.split(' ')[-1]} 100%"],
                        fontsize=9)

    overall = snap.field_tilt()
    components.header(fig, kicker="Field Tilt",
                       title=f"{name}'s final-third share so far this season: {overall:.0%}",
                       dek=f"{snap.matches_played} matches pooled by minute-of-match  ·  share of final-third touches, 5-minute buckets",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_field_tilt_over_time_{slug}.png")


# ---------------------------------------------------------------------------
# 26. Ball recoveries by third (both teams' own matches)
# ---------------------------------------------------------------------------

def recoveries_by_third(hkr, plz):
    fig, palette = new_fig()
    ax = fig.add_axes([0.24, 0.16, 0.66, 0.56])

    def by_third(snap):
        r = snap.own(snap.recoveries)
        return [sum(1 for x in r if x["x"] < 35), sum(1 for x in r if 35 <= x["x"] < 70),
                sum(1 for x in r if x["x"] >= 70)]

    boh_counts = by_third(hkr)
    hkr_counts = by_third(plz)
    zone_labels = ["Defensive third", "Middle third", "Attacking third"]
    n = len(zone_labels)
    ypos = np.arange(n)[::-1]
    maxval = max(boh_counts + hkr_counts) * 1.2 or 1
    for y, label, b, h in zip(ypos, zone_labels, boh_counts, hkr_counts):
        ax.barh(y + 0.18, b, height=0.32, color=HKR_C)
        ax.barh(y - 0.18, h, height=0.32, color=PLZ_C)
        ax.text(b + maxval * 0.015, y + 0.18, str(b), va="center", fontsize=10, color=palette["ink_primary"])
        ax.text(h + maxval * 0.015, y - 0.18, str(h), va="center", fontsize=10, color=palette["ink_primary"])
    ax.set_yticks(ypos)
    ax.set_yticklabels(zone_labels, fontsize=11.5, color=palette["ink_primary"])
    ax.set_xlim(0, maxval)
    ax.set_xlabel("Ball recoveries")
    ax.grid(axis="x")
    ax.set_axisbelow(True)

    legend_elems = [Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=HKR_C,
                            markersize=12, label=md.FIXTURE_HOME_NAME, linewidth=0),
                    Line2D([0], [0], marker="s", color=palette["surface"], markerfacecolor=PLZ_C,
                            markersize=12, label=md.FIXTURE_AWAY_NAME, linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.065), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Recoveries",
                       title="Where each side has won the ball back so far this season",
                       dek="Ball recoveries by pitch third, matches played so far pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "26_recoveries_by_third.png")


# ---------------------------------------------------------------------------
# 27. Turnovers in dangerous areas (both teams' own matches)
# ---------------------------------------------------------------------------

def turnovers_dangerous(hkr, plz):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.02, 0.10, 0.47, 0.62])
    ax2 = fig.add_axes([0.51, 0.10, 0.47, 0.62])
    pitch.draw(ax=ax1)
    pitch.draw(ax=ax2)

    for ax, snap, color, name in ((ax1, hkr, HKR_C, md.FIXTURE_HOME_NAME), (ax2, plz, PLZ_C, md.FIXTURE_AWAY_NAME)):
        t = snap.own(snap.turnovers)
        xs = [p["x"] for p in t]; ys = [p["y"] for p in t]
        pitch.scatter(xs, ys, ax=ax, s=70, color=color, alpha=0.75, edgecolors=palette["surface"],
                      linewidth=0.6, zorder=4)
        ax.set_title(f"{name} ({len(t)})", color=color, fontsize=12, fontweight="bold", family="sans-serif")

    components.header(fig, kicker="Turnovers",
                       title="Lost possession in the attacking half, season to date",
                       dek="Failed passes and Dispossessed events beyond the halfway line, own goal on the left, "
                           "attacking right, matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "27_turnovers_dangerous.png")

In [12]:
# ---------------------------------------------------------------------------
# 28-29. Crossing map -- one page per team
# ---------------------------------------------------------------------------

def crossing_map_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    crosses = [p for p in snap.own(snap.passes) if p["is_cross"] and p["end_x"] is not None]
    completed = [p for p in crosses if p["completed"]]
    incomplete = [p for p in crosses if not p["completed"]]
    for p in incomplete:
        pitch.arrows(p["x"], p["y"], p["end_x"], p["end_y"], ax=ax, color=palette["ink_muted"],
                    alpha=0.35, width=1.4, headwidth=5, headlength=5, zorder=2)
    for p in completed:
        pitch.arrows(p["x"], p["y"], p["end_x"], p["end_y"], ax=ax, color=color,
                    alpha=0.85, width=2.0, headwidth=6, headlength=6, zorder=4)

    legend_elems = [Line2D([0], [0], color=color, lw=2.2, label=f"Completed ({len(completed)})"),
                    Line2D([0], [0], color=palette["ink_muted"], lw=1.8, alpha=0.6, label=f"Incomplete ({len(incomplete)})")]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.05), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Crossing",
                       title=f"{name}: {len(crosses)} crosses so far this season",
                       dek=f"Own goal on the left, attacking right  ·  {snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_crossing_map_{slug}.png")


# ---------------------------------------------------------------------------
# 30-31. Zone 14 & half-space map -- one page per team
# ---------------------------------------------------------------------------

def zone14_halfspace_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    z0, z1, zy0, zy1 = md.ZONE14
    ax.add_patch(plt.Rectangle((z0, zy0), z1 - z0, zy1 - zy0, facecolor=CATEGORICAL_DARK[3],
                                alpha=0.18, edgecolor=CATEGORICAL_DARK[3], linewidth=1.0, zorder=1))
    for x0, x1, y0, y1 in md.HALF_SPACES:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor=color, alpha=0.10,
                                    edgecolor=color, linewidth=0.8, zorder=1))

    passes = [p for p in snap.own(snap.passes) if p["completed"] and p["end_x"] is not None]
    z14 = [p for p in passes if z0 <= p["end_x"] < z1 and zy0 <= p["end_y"] < zy1]
    hs = [p for p in passes if any(x0 <= p["end_x"] < x1 and y0 <= p["end_y"] < y1 for x0, x1, y0, y1 in md.HALF_SPACES)]
    for p in hs:
        pitch.scatter(p["end_x"], p["end_y"], ax=ax, s=50, color=color, alpha=0.7, zorder=3)
    for p in z14:
        pitch.scatter(p["end_x"], p["end_y"], ax=ax, s=70, color=CATEGORICAL_DARK[3], alpha=0.8, zorder=4)

    legend_elems = [Line2D([0], [0], marker="o", color=palette["surface"], markerfacecolor=color,
                            markersize=9, label=f"Half-space reception ({len(hs)})", linewidth=0),
                    Line2D([0], [0], marker="o", color=palette["surface"], markerfacecolor=CATEGORICAL_DARK[3],
                            markersize=9, label=f"Zone-14 reception ({len(z14)})", linewidth=0)]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.05), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Creative Zones",
                       title=f"{name}: half-space and zone-14 receptions so far this season",
                       dek=f"Completed-pass receptions in the two most dangerous central-lane zones, attacking "
                           f"right  ·  {snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_zone14_halfspace_{slug}.png")


# ---------------------------------------------------------------------------
# 32-33. Long ball targets -- one page per team
# ---------------------------------------------------------------------------

def long_balls_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.03, 0.10, 0.55, 0.62])
    pitch.draw(ax=ax)

    lb = [p for p in snap.own(snap.passes) if p["is_long_ball"] and p["completed"] and p["end_x"] is not None]
    by_player = {}
    for p in lb:
        by_player.setdefault(p["player"], []).append(p)
    for pl, pts in by_player.items():
        xs = [p["end_x"] for p in pts]; ys = [p["end_y"] for p in pts]
        pitch.scatter(xs, ys, ax=ax, s=90 + 30 * len(pts), color=color, alpha=0.7,
                      edgecolors=palette["surface"], linewidth=0.6, zorder=4)

    top = sorted(by_player.items(), key=lambda kv: -len(kv[1]))[:8]
    ax2 = fig.add_axes([0.63, 0.16, 0.33, 0.50])
    ypos = np.arange(len(top))[::-1]
    ax2.barh(ypos, [len(v) for _, v in top], color=color)
    ax2.set_yticks(ypos)
    ax2.set_yticklabels([k for k, _ in top], fontsize=10)
    ax2.set_xlabel("Completed long balls received")

    components.header(fig, kicker="Long Balls",
                       title=f"{name}: {len(lb)} completed long balls so far this season",
                       dek=f"Reception locations, own goal on the left, attacking right  ·  {snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_long_balls_{slug}.png")


# ---------------------------------------------------------------------------
# 34-35. Shot assists map -- one page per team
# ---------------------------------------------------------------------------

def shot_assists_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax = fig.add_axes([0.05, 0.10, 0.90, 0.62])
    pitch.draw(ax=ax)

    assists = [a for a in snap.assists if a["contestantId"] == snap.team_id]
    for a in assists:
        marker_c = GOOD_C if a["is_goal"] else color
        pitch.arrows(a["x"], a["y"], a["end_x"], a["end_y"], ax=ax, color=marker_c,
                    alpha=0.85, width=2.0, headwidth=6, headlength=6, zorder=4)
        pitch.scatter(a["shot_x"], a["shot_y"], ax=ax, s=60 + a["shot_xg"] * 500,
                     color=marker_c, edgecolors=palette["surface"], linewidth=0.8, zorder=5)

    legend_elems = [Line2D([0], [0], color=GOOD_C, lw=2.2, label="Assist -> goal"),
                    Line2D([0], [0], color=color, lw=2.2, label="Assist -> other shot")]
    fig.legend(handles=legend_elems, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.05), fontsize=10.5, labelcolor=palette["ink_secondary"])

    components.header(fig, kicker="Chance Creation",
                       title=f"{name}: {len(assists)} shot assists so far this season",
                       dek="Own goal on the left, attacking right  ·  a shot with no intervening teammate pass "
                           f"gets no assist credited  ·  {snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_shot_assists_{slug}.png")

In [13]:
# ---------------------------------------------------------------------------
# 36-37. Key passes leaderboard -- one page per team
# ---------------------------------------------------------------------------

def key_passes_leaderboard_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.20, 0.16, 0.72, 0.58])

    by_player = {}
    for a in snap.assists:
        if a["contestantId"] != snap.team_id:
            continue
        by_player.setdefault(a["assister"], {"n": 0, "xg": 0.0})
        by_player[a["assister"]]["n"] += 1
        by_player[a["assister"]]["xg"] += a["shot_xg"]

    top = sorted(by_player.items(), key=lambda kv: -kv[1]["xg"])[:10][::-1]
    ypos = np.arange(len(top))
    vals = [d["xg"] for _, d in top]
    ax.barh(ypos, vals, color=color)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{pl} ({d['n']})" for pl, d in top], fontsize=10.5)
    for y, (pl, d) in zip(ypos, top):
        ax.text(d["xg"] + max(vals, default=1) * 0.015, y, f"{d['xg']:.2f} xG", va="center", fontsize=9,
                color=palette["ink_secondary"])
    ax.set_xlabel("xG of shots assisted")

    components.header(fig, kicker="Chance Creation",
                       title=f"{name}: who has created the most dangerous chances so far this season",
                       dek=f"Sum of xG on shots each player assisted  ·  count in brackets = shot assists  ·  "
                           f"{snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_key_passes_{slug}.png")


# ---------------------------------------------------------------------------
# 38-39. xT flow -- one page per team, every match played so far
# ---------------------------------------------------------------------------

def xt_flow_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.08, 0.16, 0.78, 0.60])

    def series(rows):
        team_passes = sorted([p for p in rows if p["completed"]], key=lambda p: p["minute"])
        mins, cum, total = [0.0], [0.0], 0.0
        for p in team_passes:
            mins.append(p["minute"]); cum.append(total)
            total += max(0.0, p["xt_added"])
            mins.append(p["minute"]); cum.append(total)
        mins.append(96); cum.append(total)
        return mins, cum

    own_passes = snap.own(snap.passes)
    opp_passes = snap.against(snap.passes)
    for rows, color_, label in ((own_passes, color, name), (opp_passes, palette["ink_muted"], snap.opponents_label)):
        mins, cum = series(rows)
        ax.plot(mins, cum, color=color_, linewidth=2.4, zorder=4)
        ax.fill_between(mins, cum, step=None, color=color_, alpha=0.10, zorder=1)
        ax.annotate(f"{label}\n{cum[-1]:.2f} xT", xy=(1, cum[-1]), xycoords=("axes fraction", "data"),
                    xytext=(10, 0), textcoords="offset points", color=color_, fontsize=10,
                    fontweight="bold", va="center", ha="left", annotation_clip=False)

    ax.axvline(45, color=palette["axis"], linewidth=0.8, linestyle=":")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Minute of match")
    ax.set_ylabel("Cumulative xT added (completed passes)")

    own_xt = sum(max(0.0, p["xt_added"]) for p in own_passes if p["completed"])
    opp_xt = sum(max(0.0, p["xt_added"]) for p in opp_passes if p["completed"])
    verb = "have out-threatened" if own_xt > opp_xt else "have been out-threatened by"
    components.header(fig, kicker="xT Flow",
                       title=f"{name} {verb} their opponents so far this season",
                       dek=f"{snap.matches_played} matches pooled by minute-of-match  ·  own xT proxy model "
                           "(distance+angle geometry, not a possession-value model -- see match_data.py)",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_xt_flow_{slug}.png")


# ---------------------------------------------------------------------------
# 40-41. xT leaderboard -- one page per team
# ---------------------------------------------------------------------------

def xt_leaderboard_team_page(snap, color, name, page_num, slug):
    fig, palette = new_fig()
    ax = fig.add_axes([0.20, 0.16, 0.72, 0.58])

    by_player = {}
    for p in snap.own(snap.passes):
        if not p["completed"]:
            continue
        by_player[p["player"]] = by_player.get(p["player"], 0) + max(0.0, p["xt_added"])
    top = sorted(by_player.items(), key=lambda kv: -kv[1])[:10][::-1]
    ypos = np.arange(len(top))
    vals = [v for _, v in top]
    ax.barh(ypos, vals, color=color)
    ax.set_yticks(ypos)
    ax.set_yticklabels([p for p, _ in top], fontsize=10.5, color=palette["ink_primary"])
    for y, v in zip(ypos, vals):
        ax.text(v + max(vals, default=1) * 0.02, y, f"{v:.2f}", va="center", fontsize=9, color=palette["ink_secondary"])
    ax.set_xlabel("xT added")

    components.header(fig, kicker="Threat Creation",
                       title=f"{name}: who has generated the most expected threat so far this season",
                       dek=f"Sum of positive xT added by completed passes, own xT proxy model  ·  {snap.matches_played} matches pooled",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, f"{page_num}_xt_leaderboard_{slug}.png")


# ---------------------------------------------------------------------------
# 42. Shot zones heatmap (both teams' own matches)
# ---------------------------------------------------------------------------

def shot_zones_heatmap(hkr, plz):
    fig, palette = new_fig()
    pitch = new_pitch(palette)
    ax1 = fig.add_axes([0.02, 0.10, 0.47, 0.62])
    ax2 = fig.add_axes([0.51, 0.10, 0.47, 0.62])
    pitch.draw(ax=ax1)
    pitch.draw(ax=ax2)

    for ax, snap, color, name in ((ax1, hkr, HKR_C, md.FIXTURE_HOME_NAME), (ax2, plz, PLZ_C, md.FIXTURE_AWAY_NAME)):
        shots = snap.own(snap.shots)
        xs = [s["x"] for s in shots]; ys = [s["y"] for s in shots]
        cmap = "Greens" if color == HKR_C else "Blues"
        if xs:
            stats = pitch.bin_statistic(xs, ys, statistic="count", bins=(6, 4))
            pitch.heatmap(stats, ax=ax, cmap=cmap, edgecolors=palette["surface"], alpha=0.9, zorder=1)
        ax.set_title(f"{name} ({len(shots)} shots)", color=color, fontsize=12, fontweight="bold", family="sans-serif")

    components.header(fig, kicker="Shot Origin",
                       title="Where each side's shots have come from, season to date",
                       dek="Shot count density by pitch zone, matches pooled, attacking right",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "42_shot_zones_heatmap.png")

In [14]:
# ---------------------------------------------------------------------------
# 46-47. League-wide bonus pages (16-team season-to-date sample)
# ---------------------------------------------------------------------------

def pace_vs_volume_ranking(league):
    """The user's requested "m/s vs amount of passes" chart -- pass tempo
    vs total pass volume, all 16 teams with a match feed, season-to-date
    totals (2 matches each for most teams, 3 for Hradec, whose matchday-3 game has since been played). Both fixture sides highlighted."""
    fig, palette = new_fig()
    ax = fig.add_axes([0.10, 0.16, 0.82, 0.58])

    for tid, tm in league.teams.items():
        x, y = tm.pass_volume(), tm.pass_tempo_mps()
        is_boh = tid == md.HRADEC_ID
        is_hkr = tid == md.PLZEN_ID
        color = HKR_C if is_boh else (PLZ_C if is_hkr else LEAGUE_MUTED)
        ax.scatter([x], [y], s=170 if (is_boh or is_hkr) else 60, color=color,
                   edgecolors=palette["ink_primary"] if (is_boh or is_hkr) else "none", linewidth=1.6,
                   zorder=5 if (is_boh or is_hkr) else 3)
        if is_boh:
            offset = (7, 9)  # Hradec (3 matches pooled) has by far the highest pass volume -- clear of neighbours
        elif is_hkr:
            offset = (7, 9)  # Plzeň's nearest neighbour (Sigma Olomouc) sits well to its lower-left
        else:
            offset = (7, 5)
        ax.annotate(md.ALL_TEAM_NAMES[tid], xy=(x, y), xytext=offset, textcoords="offset points",
                    fontsize=9.5 if (is_boh or is_hkr) else 8,
                    color=palette["ink_primary"] if (is_boh or is_hkr) else palette["ink_muted"],
                    fontweight="bold" if (is_boh or is_hkr) else "normal")

    ax.set_xlabel("Passes completed, season to date")
    ax.set_ylabel("Pass tempo (m/s)")

    components.header(fig, kicker="Tempo",
                       title="Pace of play vs pass volume, season to date",
                       dek="Pass tempo = pass distance ÷ time to the next event (gaps >8s excluded)  ·  "
                           "16-team sample (2 matches each, 3 for Hradec)  ·  fixture sides highlighted",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "46_pace_vs_volume.png")


def verticality_ranking(league):
    fig, palette = new_fig()
    ax = fig.add_axes([0.22, 0.12, 0.70, 0.62])
    ranking = league.ranking(lambda tm: tm.verticality())
    ypos = np.arange(len(ranking))[::-1]

    def color_for(tid):
        if tid == md.HRADEC_ID:
            return HKR_C
        if tid == md.PLZEN_ID:
            return PLZ_C
        return palette["axis"]

    colors = [color_for(tid) for tid, _ in ranking]
    ax.barh(ypos, [v for _, v in ranking], color=colors)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"#{i+1}  {md.ALL_TEAM_NAMES[tid]}" for i, (tid, _) in enumerate(ranking)], fontsize=10)
    for y, (tid, v) in zip(ypos, ranking):
        weight = "bold" if tid in (md.HRADEC_ID, md.PLZEN_ID) else "normal"
        ax.text(v + max(v2 for _, v2 in ranking) * 0.015, y, f"{v:.1f}", va="center", fontsize=9.5,
                color=palette["ink_primary"], fontweight=weight)
    avg = sum(v for _, v in ranking) / len(ranking)
    ax.axvline(avg, color=palette["ink_muted"], linewidth=1.0, linestyle="--")
    ax.text(avg, len(ranking) - 0.3, " sample avg", fontsize=8, color=palette["ink_muted"], va="bottom")

    components.header(fig, kicker="League Ranking",
                       title="Team verticality, season to date",
                       dek="Avg forward distance (m) per completed forward pass  ·  16-team sample (2 matches each, 3 for Hradec)",
                       palette=palette)
    components.footer(fig, source=md.SOURCE, palette=palette)
    save(fig, "47_verticality_ranking.png")


def main():
    hkr = md.TeamSnapshot(md.HRADEC_ID)
    plz = md.TeamSnapshot(md.PLZEN_ID)
    sim = md.simulate_scorelines(hkr.own(hkr.shots), plz.own(plz.shots))
    league = md.LeagueSeason()

    cover()
    fixture_context(hkr, plz)
    shot_maps_mw1(hkr, plz)
    xg_snapshot(hkr, plz)
    xg_flow_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "05", "hradec")
    xg_flow_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "06", "plzen")
    shot_quality_table_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "07", "hradec")
    shot_quality_table_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "08", "plzen")
    goal_buildups_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "09", "hradec")
    goal_buildups_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "10", "plzen")
    pass_network_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "11", "hradec")
    pass_network_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "12", "plzen")
    progressive_passes_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "13", "hradec")
    progressive_passes_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "14", "plzen")
    passing_directness_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "15", "hradec")
    passing_directness_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "16", "plzen")
    touch_heatmap_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "17", "hradec", "Greens")
    touch_heatmap_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "18", "plzen", "Blues")
    possession_thirds(hkr, plz)
    progression_bars(hkr, plz)
    field_tilt_over_time_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "21", "hradec")
    field_tilt_over_time_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "22", "plzen")
    ppda_pressing(hkr, plz)
    defensive_actions(hkr, plz)
    duels_discipline(hkr, plz)
    recoveries_by_third(hkr, plz)
    turnovers_dangerous(hkr, plz)
    crossing_map_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "28", "hradec")
    crossing_map_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "29", "plzen")
    zone14_halfspace_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "30", "hradec")
    zone14_halfspace_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "31", "plzen")
    long_balls_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "32", "hradec")
    long_balls_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "33", "plzen")
    shot_assists_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "34", "hradec")
    shot_assists_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "35", "plzen")
    key_passes_leaderboard_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "36", "hradec")
    key_passes_leaderboard_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "37", "plzen")
    xt_flow_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "38", "hradec")
    xt_flow_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "39", "plzen")
    xt_leaderboard_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "40", "hradec")
    xt_leaderboard_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "41", "plzen")
    shot_zones_heatmap(hkr, plz)
    key_players_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "43", "hradec")
    key_players_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "44", "plzen")
    team_radar(hkr, plz)
    pace_vs_volume_ranking(league)
    verticality_ranking(league)
    keys_to_the_game(hkr, plz)
    win_probability(hkr, plz, sim)
    report_card(hkr, plz, sim)
    print("Done.")


if __name__ == "__main__":
    main()

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/01_cover.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/02_fixture_context.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/03_shot_maps_mw1.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/04_xg_snapshot.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/05_xg_flow_hradec.png


...

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/46_pace_vs_volume.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/47_verticality_ranking.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/48_keys_to_the_game.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/49_win_probability.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/50_report_card.png
Done.


## Generate all 50 pages and compile the PDF

In [15]:
def main():
    hkr = TeamSnapshot(md.HRADEC_ID)
    plz = TeamSnapshot(md.PLZEN_ID)
    sim = simulate_scorelines(hkr.own(hkr.shots), plz.own(plz.shots))
    league = LeagueSeason()

    cover()
    fixture_context(hkr, plz)
    shot_maps_mw1(hkr, plz)
    xg_snapshot(hkr, plz)
    xg_flow_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "05", "hradec")
    xg_flow_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "06", "plzen")
    shot_quality_table_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "07", "hradec")
    shot_quality_table_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "08", "plzen")
    goal_buildups_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "09", "hradec")
    goal_buildups_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "10", "plzen")
    pass_network_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "11", "hradec")
    pass_network_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "12", "plzen")
    progressive_passes_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "13", "hradec")
    progressive_passes_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "14", "plzen")
    passing_directness_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "15", "hradec")
    passing_directness_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "16", "plzen")
    touch_heatmap_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "17", "hradec", "Greens")
    touch_heatmap_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "18", "plzen", "Blues")
    possession_thirds(hkr, plz)
    progression_bars(hkr, plz)
    field_tilt_over_time_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "21", "hradec")
    field_tilt_over_time_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "22", "plzen")
    ppda_pressing(hkr, plz)
    defensive_actions(hkr, plz)
    duels_discipline(hkr, plz)
    recoveries_by_third(hkr, plz)
    turnovers_dangerous(hkr, plz)
    crossing_map_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "28", "hradec")
    crossing_map_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "29", "plzen")
    zone14_halfspace_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "30", "hradec")
    zone14_halfspace_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "31", "plzen")
    long_balls_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "32", "hradec")
    long_balls_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "33", "plzen")
    shot_assists_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "34", "hradec")
    shot_assists_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "35", "plzen")
    key_passes_leaderboard_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "36", "hradec")
    key_passes_leaderboard_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "37", "plzen")
    xt_flow_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "38", "hradec")
    xt_flow_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "39", "plzen")
    xt_leaderboard_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "40", "hradec")
    xt_leaderboard_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "41", "plzen")
    shot_zones_heatmap(hkr, plz)
    key_players_team_page(hkr, HKR_C, md.FIXTURE_HOME_NAME, "43", "hradec")
    key_players_team_page(plz, PLZ_C, md.FIXTURE_AWAY_NAME, "44", "plzen")
    team_radar(hkr, plz)
    pace_vs_volume_ranking(league)
    verticality_ranking(league)
    keys_to_the_game(hkr, plz)
    win_probability(hkr, plz, sim)
    report_card(hkr, plz, sim)
    print("Done.")


In [16]:
main()  # build_charts.main() -- generates all 50 PNGs into ./Visuals

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/01_cover.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/02_fixture_context.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/03_shot_maps_mw1.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/04_xg_snapshot.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/05_xg_flow_hradec.png


...

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/46_pace_vs_volume.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/47_verticality_ranking.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/48_keys_to_the_game.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/49_win_probability.png


Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Visuals/50_report_card.png
Done.


## Compile the PDF (`build_pdf.py`)

In [17]:
import glob
import os

from PIL import Image
from reportlab.lib.pagesizes import landscape
from reportlab.pdfgen import canvas

OUT_DIR = NOTEBOOK_DIR
VIS_DIR = os.path.join(OUT_DIR, "Visuals")
PDF_PATH = os.path.join(OUT_DIR, "Hradec_Kralove_vs_Viktoria_Plzen_PreMatch.pdf")

PAGE_W, PAGE_H = 1920, 1080  # points, 16:9


def build_pdf():
    pages = sorted(glob.glob(os.path.join(VIS_DIR, "*.png")))
    if not pages:
        raise SystemExit("No PNGs found in Visuals/ -- run build_charts.py first")

    c = canvas.Canvas(PDF_PATH, pagesize=landscape((PAGE_H, PAGE_W)))
    for path in pages:
        img = Image.open(path)
        iw, ih = img.size
        scale = min(PAGE_W / iw, PAGE_H / ih)
        w, h = iw * scale, ih * scale
        x, y = (PAGE_W - w) / 2, (PAGE_H - h) / 2
        c.drawImage(path, x, y, width=w, height=h)
        c.showPage()
    c.save()
    print("Saved:", PDF_PATH, f"({len(pages)} pages)")

In [18]:
build_pdf()  # compiles Visuals/*.png into the landscape PDF

Saved: /home/user/eredivisienanalytics/CZ Events/CZ 2026-2027 Reports/Hradec Kralove vs Viktoria Plzen/Hradec_Kralove_vs_Viktoria_Plzen_PreMatch.pdf (50 pages)
